# Monarch KG → Knowledge Graph (KG) Builder

**Source:** [Monarch Initiative KG](https://monarchinitiative.org/) —

## What this notebook does

1. Loads all reference dictionaries (NCBI, PubChem, ChEBI, UniProt, DO, MONDO).
2. Loads the full Monarch KG edge pickle, strips NCBITaxon prefixes, maps taxon IDs to species names.
3. Filters to 8 EvoAge target species; removes 44 out-of-scope relation types.
4. Processes **55 active relation types** and exports per-relation, per-species CSV files.

## Species retained
- *Homo sapiens*, *Mus musculus*, *Danio rerio*, *Rattus norvegicus*
- *Drosophila melanogaster*, *Caenorhabditis elegans*
- *Saccharomyces cerevisiae S288C*, *Saccharomyces cerevisiae*

## Output directory structure
```
Monarch/
├── Human/        ← Human-specific outputs
├── Mouse/        ← Mouse-specific outputs
├── Celegans/     ← C. elegans-specific outputs
├── Drosophila/   ← Drosophila-specific outputs
├── Zebrafish/    ← Zebrafish-specific outputs
├── Rat/          ← Rat-specific outputs
└── Yeast/        ← Yeast-specific outputs
```



# PPRE Processing 

In [18]:
# Edges = pd.read_csv('/storage/Arushi/090526_EvoAge/kg_formation/data_collection/monarchkg/monarch-kg_edges.tsv', sep='\t')
# Edges

In [19]:
# import pandas as pd

# # Load nodes — contains id, name, category, in_taxon, in_taxon_label, symbol
# Nodes = pd.read_csv('/storage/Arushi/090526_EvoAge/kg_formation/data_collection/monarchkg/monarch-kg_nodes.tsv', sep='\t',
#                     usecols=['id','category','name','in_taxon','in_taxon_label'],
#                     dtype=str)

# # Build lookup dicts from node table
# node_name_dict   = dict(zip(Nodes['id'], Nodes['name']))
# node_cat_dict    = dict(zip(Nodes['id'], Nodes['category']))
# node_taxon_dict  = dict(zip(Nodes['id'], Nodes['in_taxon']))
# node_taxlab_dict = dict(zip(Nodes['id'], Nodes['in_taxon_label']))

# # Load edges — minimal columns only
# Edges = pd.read_csv('/storage/Arushi/090526_EvoAge/kg_formation/data_collection/monarchkg/monarch-kg_edges.tsv', sep='\t',
#                     usecols=['subject','object','predicate',
#                              'subject_category','object_category',
#                              'primary_knowledge_source'],
#                     dtype=str)

# # Strip biolink: prefix
# for col in ['subject_category','object_category','predicate']:
#     Edges[col] = Edges[col].str.replace('biolink:','', regex=False)

# # Denormalize: join node labels/taxons onto edges
# Edges['subject_label']      = Edges['subject'].map(node_name_dict)
# Edges['subject_namespace']  = Edges['subject'].str.split(':').str[0]
# Edges['subject_taxon']      = Edges['subject'].map(node_taxon_dict)
# Edges['subject_taxon_label']= Edges['subject'].map(node_taxlab_dict)

# Edges['object_label']       = Edges['object'].map(node_name_dict)
# Edges['object_namespace']   = Edges['object'].str.split(':').str[0]
# Edges['object_taxon']       = Edges['object'].map(node_taxon_dict)
# Edges['object_taxon_label'] = Edges['object'].map(node_taxlab_dict)

# # Build Relation column
# Edges['Relation'] = (Edges['subject_category'].str.replace(' ','') + '_' +
#                      Edges['object_category'].str.replace(' ',''))

# # Save as pickle for fast reload in processing notebooks
# Edges.to_pickle('monarch-kg-denormalized-edges_Arushi.pkl')
# print(f"Saved {len(Edges):,} rows")

In [20]:
# Edges = pd.read_pickle('monarch-kg-denormalized-edges_Arushi.pkl')
# Edges

In [34]:
# ---
## 0 · Configuration
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# USER CONFIGURATION — set the data folder before running
# ─────────────────────────────────────────────────────────────────────────────
DATA_DIR = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/monarchkg/"

import os
# ASSOC_PATH  = os.path.join(DATA_DIR, "association.all.tsv")
NODES_PATH  = os.path.join(DATA_DIR, "monarch-kg-denormalized-nodes.tsv")
EDGES_TSV   = os.path.join(DATA_DIR, "monarch-kg-denormalized-edges.tsv")
EDGES_PKL   = os.path.join(DATA_DIR, "monarch-kg-denormalized-edges.pkl")
RELATION_CSV= os.path.join(DATA_DIR, "Relation_type_count.csv")

print("Paths set.")
# ---
## 1 · Curated association file

# Contains genes, drugs, and diseases with 29 relation types.  
# Used for initial scoping — the full edge TSV is processed separately below.
# Load the curated association file
# This is a smaller, pre-filtered view of Monarch associations
# association = pd.read_csv(ASSOC_PATH, sep='\t')

# print(f"Association file: {len(association):,} rows x {association.shape[1]} columns")
# print("Relation types present:", association['predicate'].nunique() if 'predicate' in association.columns else "N/A")
# association.head(3)
# # ---
## 2 · Node table inspection

# Used to understand which ID namespaces (DB_id prefixes) are present in Monarch.
# Load the full Monarch node table
# Nodes = pd.read_csv(NODES_PATH, sep='\t')

# Split the 'id' column on ':' to extract the database prefix (e.g. 'HGNC', 'MONDO', 'GO')
# and the numeric/alphanumeric identifier separately
# Nodes[['DB_id', 'ID']] = Nodes['id'].str.split(':', n=1, expand=True)

# print(f"Nodes table: {len(Nodes):,} rows x {Nodes.shape[1]} columns")
# print("\nTop 20 ID namespace prefixes (DB_id):")
# print(Nodes['DB_id'].value_counts().head(20))
# ---
## 3 · Edge table — inspect a sample first

# The full TSV is ~12.7M rows. We read 5 rows first to confirm column names before loading everything.
# Preview 5 rows to confirm column names and data format
# Total edge count is approximately 957,628 + 11,701,549 = 12,659,177 rows
Edges_5 = pd.read_csv(EDGES_TSV, sep='\t', nrows=5)

print("Edge table columns:")
print(list(Edges_5.columns))
print(f"\nSample rows:")
Edges_5
# ---
## 4 · Load full edge table — required columns only

# Reading only the needed columns reduces memory from ~40GB to a manageable size.
# Only these columns are needed for KG construction.
# 'subject_closure' and 'object_closure' carry ancestor term lists
# and are included for potential downstream ontology expansion.
required_columns = [
    'predicate',
    'subject', 'object',
    'subject_label', 'subject_category', 'subject_namespace',
    'subject_closure', 'subject_closure_label',
    'subject_taxon', 'subject_taxon_label',
    'object_label', 'object_category', 'object_namespace',
    'object_closure', 'object_closure_label',
    'object_taxon', 'object_taxon_label',
    'primary_knowledge_source'
]

print("Loading full Monarch edge table (this may take several minutes)...")
Edges = pd.read_csv(
    EDGES_TSV,
    sep='\t',
    usecols=required_columns,
    dtype=str           # load as string to avoid mixed-type inference issues
)
print(f"Loaded: {len(Edges):,} rows x {Edges.shape[1]} columns")
# ---
## 5 · Strip `biolink:` prefix from category and predicate columns

# All `subject_category`, `object_category`, and `predicate` values are prefixed with `biolink:` in the raw file.  
# Stripping this gives cleaner values like `Gene`, `Disease`, `interacts_with`.
# Strip 'biolink:' prefix from the three columns that use it
for col in ['subject_category', 'object_category', 'predicate']:
    Edges[col] = Edges[col].str.replace('biolink:', '', regex=False)

print("Prefix stripped. Sample subject_category values:")
print(Edges['subject_category'].value_counts().head(10))
# ---
## 6 · Build the `Relation` column

# Combines Head entity type and Tail entity type into a single relation string  
# (e.g. `Gene_Disease`, `Gene_BiologicalProcess`) for fast filtering in the processing notebook.
# Combine subject and object categories into a Relation string
# Spaces are removed so e.g. 'Molecular Activity' becomes 'MolecularActivity'
Edges['Relation'] = (
    Edges['subject_category'].str.replace(' ', '', regex=False) + '_' +
    Edges['object_category'].str.replace(' ', '', regex=False)
)

print("Relation column built.")
print(f"Unique relation types: {Edges['Relation'].nunique()}")
print("\nTop 20 relation types by count:")
print(Edges['Relation'].value_counts().head(20))

# ---
## 7 · Inspect knowledge sources and specific entity types
# Distribution of primary knowledge sources (which databases contributed each edge)
print("Knowledge source distribution:")
print(Edges['primary_knowledge_source'].value_counts().head(15))
# Inspect namespace distribution across both Head and Tail nodes
# Useful for understanding which ontology IDs are present
combined_ns = pd.concat([Edges['subject_namespace'], Edges['object_namespace']])
ns_counts = combined_ns.value_counts().reset_index()
ns_counts.columns = ['Namespace', 'Count']
print("Top 30 entity ID namespaces (combined Head + Tail):")
ns_counts.head(30)
# Quick look at specific entity types of interest
print("Rows where object is a Protein:", len(Edges[Edges['object_category'] == 'Protein']))
print("Rows where object is a PhenotypicFeature:", len(Edges[Edges['object_category'] == 'PhenotypicFeature']))

# ---
## 8 · Save processed edges to pickle

# Pickle preserves dtypes and loads ~10x faster than TSV on subsequent runs.
# Save the processed edge table as a pickle for fast reloading in processing notebooks
Edges.to_pickle(EDGES_PKL)
print(f"Saved processed edges to: {EDGES_PKL}")
print(f"File size: {os.path.getsize(EDGES_PKL) / 1e9:.2f} GB")

# ---
## 9 · Export Relation type counts

# Used to decide which relation types to include or exclude in the KG processing notebook.
# Count occurrences of each Relation type and export for manual review
relation_counts = Edges['Relation'].value_counts().reset_index()
relation_counts.columns = ['Relation', 'Count']

relation_counts.to_csv(RELATION_CSV, index=False)
print(f"Saved relation counts to: {RELATION_CSV}")
print(f"Total relation types: {len(relation_counts)}")
relation_counts

Paths set.
Edge table columns:
['id', 'original_subject', 'predicate', 'original_object', 'category', 'agent_type', 'aggregator_knowledge_source', 'knowledge_level', 'primary_knowledge_source', 'publications', 'provided_by', 'disease_context_qualifier', 'frequency_qualifier', 'has_count', 'has_percentage', 'has_quotient', 'has_total', 'has_evidence', 'qualifiers', 'stage_qualifier', 'original_predicate', 'subject_specialization_qualifier', 'negated', 'onset_qualifier', 'sex_qualifier', 'species_context_qualifier', 'subject', 'object', 'subject_label', 'subject_category', 'subject_namespace', 'subject_closure', 'subject_closure_label', 'subject_taxon', 'subject_taxon_label', 'object_label', 'object_category', 'object_namespace', 'object_closure', 'object_closure_label', 'object_taxon', 'object_taxon_label', 'disease_context_qualifier_label', 'disease_context_qualifier_category', 'disease_context_qualifier_namespace', 'disease_context_qualifier_closure', 'disease_context_qualifier_closur

,Relation,Count
0,Gene_Gene,3515916
1,Gene_AnatomicalEntity,2260542
2,Gene_BiologicalProcess,1039377
3,Gene_PhenotypicFeature,964816
4,Gene_MolecularActivity,933964
...,...,...
110,MolecularEntity_ChemicalEntity,2
111,Protein_NamedThing,1
112,CellularComponent_AnatomicalEntity,1
113,MolecularActivity_NamedThing,1


---
## 0 · Configuration — edit ONLY these two lines

In [8]:
import os
import re
import numpy as np
import pandas as pd
from glob import glob

# ─────────────────────────────────────────────────────────────────────────────
# USER CONFIGURATION
# BASE_PATH : root folder containing all raw input data
# OUT_PATH  : root folder where all processed CSVs will be saved
# ─────────────────────────────────────────────────────────────────────────────
BASE_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/"
OUT_PATH  = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data/MonarchNew/"

# ── Derived input paths (do not edit below this line) ────────────────────────
ENS2NCBI_PATH     = os.path.join(BASE_PATH, "databases_for_mapping/ENSEMBL/ENSEMBLE_ID_2_NCBI_Gene_SYM.csv")
NCBI_HUMAN_PATH   = os.path.join(BASE_PATH, "databases_for_mapping/ncbi/Homo_sapiens.gene_info")
PUBCHEM_PKL_PATH  = os.path.join(BASE_PATH, "databases_for_mapping/pubchem/combined_df.pkl")
PUBCHEM_DB_PATH   = os.path.join(BASE_PATH, "databases_for_mapping/pubchem/Pubchem_ID_DrugBank_Chebi.csv")
DO_PATH           = os.path.join(BASE_PATH, "databases_for_mapping/DO/All_DO_ref_files.csv")
MONDO_PATH        = os.path.join(BASE_PATH, "databases_for_mapping/DO/MONDO_ID_Name_coll_from_Monarch.csv")
UNIPROT_MAP_PATH  = os.path.join(BASE_PATH, "databases_for_mapping/uniprot/UNIPROT_NCBI_ALL_MAPPED.csv")
UNIPROT_REC_PATH  = os.path.join(BASE_PATH, "databases_for_mapping/uniprot/Uniprot_ID_Recname.csv")
MONARCH_PKL_PATH  = os.path.join(BASE_PATH, "monarchkg/monarch-kg-denormalized-edges.pkl")

# ── Output sub-directories (one per species) ──────────────────────────────────
OUT_HUMAN  = os.path.join(OUT_PATH, "Human/")
OUT_MOUSE  = os.path.join(OUT_PATH, "Mouse/")
OUT_CELE   = os.path.join(OUT_PATH, "Celegans/")
OUT_DROSO  = os.path.join(OUT_PATH, "Drosophila/")
OUT_ZEBRA  = os.path.join(OUT_PATH, "Zebrafish/")
OUT_RAT    = os.path.join(OUT_PATH, "Rat/")
OUT_YEAST  = os.path.join(OUT_PATH, "Yeast/")

for d in [OUT_HUMAN, OUT_MOUSE, OUT_CELE, OUT_DROSO, OUT_ZEBRA, OUT_RAT, OUT_YEAST]:
    os.makedirs(d, exist_ok=True)

print("Paths configured successfully.")
print(f"  Input  root : {BASE_PATH}")
print(f"  Output root : {OUT_PATH}")

Paths configured successfully.
  Input  root : /storage/Arushi/090526_EvoAge/kg_formation/data_collection/
  Output root : /storage/Arushi/090526_EvoAge/kg_formation/processed_data/MonarchNew/


---
## 1 · Load reference dictionaries

In [2]:
# ── 1a. ENSEMBL <-> NCBI gene symbol crosswalk ───────────────────────────────
ENS_2NCBI = pd.read_csv(ENS2NCBI_PATH)
ENS_2NCBI = ENS_2NCBI[~ENS_2NCBI['name'].isna()]
NCBI_2ENS__dict = dict(zip(ENS_2NCBI['name'], ENS_2NCBI['initial_alias']))  # {symbol: ENS_ID}
del ENS_2NCBI
print(f"Symbol -> Ensembl: {len(NCBI_2ENS__dict):,}")

Symbol -> Ensembl: 40,940


In [3]:
# ── 1b. NCBI Human gene_info ──────────────────────────────────────────────────
# Monarch uses gene symbols (strings) as gene IDs, not GeneIDs (integers).
# NCBI_ALL_GENEname_dict: Symbol -> description
NCBI_ALL_GENE = pd.read_csv(NCBI_HUMAN_PATH, sep='\t')
NCBI_ALL_GENE['ENSEMBLE_ID'] = NCBI_ALL_GENE['Symbol'].map(NCBI_2ENS__dict)
NCBI_ALL_GENEname_dict   = dict(zip(NCBI_ALL_GENE['Symbol'],  NCBI_ALL_GENE['description']))
NCBI_ALL_GENEIDname_dict = dict(zip(NCBI_ALL_GENE['GeneID'],  NCBI_ALL_GENE['Symbol']))

# HGNC ID -> Symbol (used for HGNC-prefixed gene IDs in Monarch)
def extract_hgnc(value):
    match = re.search(r'HGNC:(HGNC:\d+)', str(value))
    return match.group(1) if match else None
NCBI_ALL_GENE['HGNC'] = NCBI_ALL_GENE['dbXrefs'].apply(extract_hgnc)
NCBI_ALL_GENE_HGNC = NCBI_ALL_GENE.dropna(subset=['HGNC'])
NCBI_ALL_GENE_HGNC_dict = dict(zip(NCBI_ALL_GENE_HGNC['HGNC'], NCBI_ALL_GENE_HGNC['Symbol']))

print(f"NCBI Human genes: {len(NCBI_ALL_GENEname_dict):,}")
print(f"HGNC -> Symbol:   {len(NCBI_ALL_GENE_HGNC_dict):,}")

NCBI Human genes: 193,352
HGNC -> Symbol:   44,076


In [4]:
# ── 1c. Gene synonym map — built ONCE globally ────────────────────────────────

# Each rebuild is a full scan of NCBI_ALL_GENE. Built once here, reused everywhere.
synonym_map = {}
for _, row in NCBI_ALL_GENE.iterrows():
    for syn in row['Synonyms'].split('|'):
        synonym_map[syn.strip()] = row['Symbol']
print(f"Gene synonym map: {len(synonym_map):,} aliases")

Gene synonym map: 69,564 aliases


In [5]:
# ── 1d. PubChem SMILES -> CID ─────────────────────────────────────────────────
Pubchem = pd.read_pickle(PUBCHEM_PKL_PATH)
Pubchem_Smile_CID_Dict = dict(zip(Pubchem['PUBCHEM_SMILES'], Pubchem['PUBCHEM_COMPOUND_CID']))
del Pubchem
print(f"PubChem SMILES -> CID: {len(Pubchem_Smile_CID_Dict):,}")

PubChem SMILES -> CID: 119,125,801


In [6]:
# ── 1e. DrugBank -> PubChem CID and ChEBI -> PubChem CID ─────────────────────
# Monarch chemical entities use ChEBI IDs; resolved to PubChem CID via this crosswalk
pubchem = pd.read_csv(PUBCHEM_DB_PATH)
pubchem_DB    = pubchem[~pubchem['DRUGBANK_ID'].isna()]
pubchem_CHEBI = pubchem[~pubchem['CHEBI_ID'].isna()]
DB2PC_dict    = dict(zip(pubchem_DB['DRUGBANK_ID'],    pubchem_DB['ID']))    # {DB_ID: CID}
Chebi2PC_dict = dict(zip(pubchem_CHEBI['CHEBI_ID'],   pubchem_CHEBI['ID'])) # {CHEBI:XXXXX: CID}
print(f"DrugBank -> PubChem: {len(DB2PC_dict):,} | ChEBI -> PubChem: {len(Chebi2PC_dict):,}")

/tmp/ipykernel_2070228/404589850.py:3: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  pubchem = pd.read_csv(PUBCHEM_DB_PATH)


DrugBank -> PubChem: 10,702 | ChEBI -> PubChem: 177,680


In [7]:
# ── 1f. Disease Ontology (DO): label -> DOID ──────────────────────────────────
# NOTE: in this notebook DO dict is label->id (reverse of other notebooks)
# because Monarch disease names need to be resolved to DOID
DO_All_File = pd.read_csv(DO_PATH)
DOID_disname_dict  = dict(zip(DO_All_File['label'], DO_All_File['id']))
DOID_disAltID_dict = dict(zip(DO_All_File['label'], DO_All_File['xrefs']))
print(f"DO entries: {len(DOID_disname_dict):,}")

DO entries: 11,787


In [9]:
# ── 1g. MONDO: name -> MONDO ID ──────────────────────────────────────────────
# Fallback for disease names not found in DO
MONDO = pd.read_csv(MONDO_PATH)
MONDO_dict = dict(zip(MONDO['NAME'], MONDO['ID']))
print(f"MONDO entries: {len(MONDO_dict):,}")

MONDO entries: 25,926


In [10]:
# ── 1h. UniProt: RecName -> AC and Gene_Name -> UniProt_ID ───────────────────
Uniprot_names = pd.read_csv(UNIPROT_MAP_PATH)
Uniprot_gene_name = Uniprot_names[~Uniprot_names['Gene_Name'].isna()].drop_duplicates(subset=['Gene_Name'], keep='first')
Uniprot_gene_name_dict = dict(zip(Uniprot_gene_name['Gene_Name'], Uniprot_gene_name['Uniprot_ID']))

Uniprot_Recname = pd.read_csv(UNIPROT_REC_PATH)
Uniprot_Recname_dict = dict(zip(Uniprot_Recname['RecName'], Uniprot_Recname['AC']))
print(f"UniProt Gene_Name -> ID: {len(Uniprot_gene_name_dict):,}")
print(f"UniProt RecName -> AC:   {len(Uniprot_Recname_dict):,}")

/tmp/ipykernel_2070228/1931560169.py:2: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  Uniprot_names = pd.read_csv(UNIPROT_MAP_PATH)


UniProt Gene_Name -> ID: 26,176
UniProt RecName -> AC:   149,154


---
## 2 · Helper functions

In [11]:
def refining_columns(df):
    """
    Rename Monarch KG columns to EvoAge KG schema and add KGSource tag.
    BUG FIX: original included 'Relation' in desired_columns but that column
    doesn't exist in the raw pickle (only subject/object/predicate) -> KeyError.
    'Relation' is excluded here and constructed AFTER loading.
    """
    df = df.rename(columns={
        'subject':                'Head',
        'object':                 'Tail',
        'predicate':              'Relation_type',
        'primary_knowledge_source': 'Relation_Source',
        'subject_label':          'Head_name',
        'subject_category':       'Head_type',
        'subject_namespace':      'Head_ID_IS',
        'subject_taxon':          'Head_taxon',
        'subject_taxon_label':    'Head_taxon_label',
        'object_label':           'Tail_name',
        'object_category':        'Tail_type',
        'object_namespace':       'Tail_ID_IS',
        'object_taxon':           'Tail_taxon',
        'object_taxon_label':     'Tail_taxon_label',
    })
    df['KGSource'] = 'MonarchKG'
    desired_columns = [
        'Head', 'Tail', 'Relation_type', 'Relation_Source', 'KGSource',
        'Head_name', 'Head_type', 'Head_ID_IS', 'Head_taxon', 'Head_taxon_label',
        'Tail_name', 'Tail_type', 'Tail_ID_IS', 'Tail_taxon', 'Tail_taxon_label'
    ]
    return df[[c for c in desired_columns if c in df.columns]]


def resolve_chebi_to_pubchem(df, col):
    """Resolve ChEBI ID column -> PubChem CID. Drops rows where no CID found."""
    df = df.copy()
    id_col = f'{col}_ID'
    df[id_col] = df[col].map(Chebi2PC_dict)
    before = len(df)
    df = df[~df[id_col].isna()]
    df[id_col] = df[id_col].astype(str).str.replace(r'\.0$', '', regex=True)
    df[[col, id_col]] = df[[id_col, col]]   # swap: CID -> col, CHEBI -> id_col
    print(f"  ChEBI {col} -> PubChem: {len(df):,} kept / {before - len(df):,} dropped")
    return df


def resolve_disease_head(df):
    """Resolve disease Head name -> DOID (preferred) or MONDO ID (fallback).
    BUG FIX: uses None not np.nan in np.where to avoid DTypePromotionError."""
    df = df.copy()
    df['Head_DO_name'] = (df['Head_name'].map(DOID_disname_dict)
                          .fillna(df['Head_name'].map(MONDO_dict))
                          .fillna(df['Head']))
    df[['Head', 'Head_DO_name']] = df[['Head_DO_name', 'Head']]
    df['Head_ID_IS'] = np.where(
        df['Head'].isna(), None,
        np.where(df['Head'].str.startswith('DOID'), 'DOID', 'MONDO'))
    return df


def resolve_gene_symbol_head(df):
    """Apply synonym map to Head gene name -> canonical NCBI Symbol."""
    df = df.copy()
    df['Head_Alias_mapped'] = df['Head_name'].apply(lambda x: synonym_map.get(x, x))
    df['Head_Detail_name']  = df['Head_Alias_mapped'].map(NCBI_ALL_GENEname_dict)
    df = df.rename(columns={'Head': 'Head_Orignal'})
    df[['Head_name', 'Head_Alias_mapped']] = df[['Head_Alias_mapped', 'Head_name']]
    df = df.rename(columns={'Head_Alias_mapped': 'Head'})
    df['KG_Source'] = 'Monarch_KG'
    return df


def resolve_gene_symbol_tail(df):
    """Apply synonym map to Tail gene name -> canonical NCBI Symbol."""
    df = df.copy()
    df['Tail_Alias_mapped'] = df['Tail_name'].apply(lambda x: synonym_map.get(x, x))
    df['Tail_Detail_name']  = df['Tail_Alias_mapped'].map(NCBI_ALL_GENEname_dict)
    df[['Tail_name', 'Tail_Alias_mapped']] = df[['Tail_Alias_mapped', 'Tail_name']]
    df = df.rename(columns={'Tail_Alias_mapped': 'Tail'})
    return df


DESIRED_COLS = [
    'Head', 'Relation', 'Tail', 'Head_type', 'Relation_type', 'Tail_type', 'Relation_Source',
    'KG_Source', 'Head_Detail_name', 'Tail_Detail_name',
    'Head_taxon_name', 'Tail_taxon_name',
    'Head_ID_IS', 'Tail_ID_IS', 'Head_name', 'Tail_name',
    'Head_Orignal', 'Species_Species'
]

def select_cols(df):
    return df[[c for c in DESIRED_COLS if c in df.columns]]

def save(df, filepath):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    df.to_csv(filepath, index=False)
    print(f"Saved {len(df):,} rows -> {filepath}")

print("Helper functions defined.")

Helper functions defined.


---
## 3 · Load and filter Monarch KG edges

In [12]:
# ── 3a. Load edge pickle and apply column renaming ───────────────────────────
Edges_raw = pd.read_pickle(MONARCH_PKL_PATH)
Edges = refining_columns(Edges_raw)
del Edges_raw

# ── 3b. Strip NCBITaxon: prefix and map taxon IDs to species names ────────────
Edges['Head_taxon'] = Edges['Head_taxon'].str.replace('NCBITaxon:', '', regex=False)
Edges['Tail_taxon'] = Edges['Tail_taxon'].str.replace('NCBITaxon:', '', regex=False)

taxon_dict = {
    9606:   'Homo sapiens',
    10090:  'Mus musculus',
    7955:   'Danio rerio',
    10116:  'Rattus norvegicus',
    7227:   'Drosophila melanogaster',
    6239:   'Caenorhabditis elegans',
    9913:   'Bos taurus',
    4896:   'Schizosaccharomyces pombe',
    8364:   'Xenopus tropicalis',
    9823:   'Sus scrofa',
    9615:   'Canis lupus familiaris',
    9031:   'Gallus gallus',
    559292: 'Saccharomyces cerevisiae S288C',
    44689:  'Dictyostelium discoideum',
    8355:   'Xenopus laevis',
    227321: 'Aspergillus nidulans',
    4932:   'Saccharomyces cerevisiae'
}
Edges['Head_taxon_name'] = pd.to_numeric(Edges['Head_taxon'], errors='coerce').map(taxon_dict)
Edges['Tail_taxon_name'] = pd.to_numeric(Edges['Tail_taxon'], errors='coerce').map(taxon_dict)

# Build the Relation column from entity type prefix
Edges['Head_type_clean'] = Edges['Head_type'].str.split(':').str[-1]
Edges['Tail_type_clean'] = Edges['Tail_type'].str.split(':').str[-1]
Edges['Relation'] = (Edges['Head_type_clean'] + '_' + Edges['Tail_type_clean']).str.replace(' ', '')

print(f"Total edges loaded: {len(Edges):,}")

Total edges loaded: 11,701,549


In [13]:
# ── 3c. Filter to 8 EvoAge target species ────────────────────────────────────
species = [
    'Caenorhabditis elegans', 'Danio rerio', 'Drosophila melanogaster',
    'Homo sapiens', 'Mus musculus', 'Saccharomyces cerevisiae S288C', 'Saccharomyces cerevisiae'
]
Edges = Edges[
    (Edges['Head_taxon_name'].isin(species) | Edges['Head_taxon_name'].isna()) &
    (Edges['Tail_taxon_name'].isin(species) | Edges['Tail_taxon_name'].isna())
]
print(f"After species filter: {len(Edges):,}")

After species filter: 9,155,273


In [14]:
# ── 3d. Remove out-of-scope relation types ────────────────────────────────────
relations_to_remove = [
    'Gene_Cell', 'NamedThing_NamedThing', 'Protein_OrganismTaxon', 'PhenotypicFeature_NamedThing',
    'AnatomicalEntity_LifeStage', 'AnatomicalEntity_NamedThing', 'NamedThing_AnatomicalEntity',
    'Cell_Cell', 'PhenotypicFeature_Cell', 'Disease_OrganismTaxon', 'OrganismTaxon_OrganismTaxon',
    'NamedThing_PhenotypicFeature', 'Cell_Protein', 'Cell_AnatomicalEntity', 'BiologicalProcess_Cell',
    'AnatomicalEntity_Cell', 'NamedThing_BiologicalProcess', 'AnatomicalEntity_OrganismTaxon',
    'LifeStage_LifeStage', 'NamedThing_Cell', 'BiologicalProcess_OrganismTaxon', 'Cell_BiologicalProcess',
    'MolecularEntity_OrganismTaxon', 'CellularComponent_OrganismTaxon', 'Cell_NamedThing',
    'NamedThing_ChemicalEntity', 'NamedThing_CellularComponent', 'CellularComponent_Cell',
    'Cell_CellularComponent', 'BiologicalProcess_NamedThing', 'Disease_Cell', 'NamedThing_Disease',
    'NamedThing_Protein', 'Disease_NamedThing', 'Cell_OrganismTaxon', 'NamedThing_MolecularActivity',
    'MolecularActivity_OrganismTaxon', 'LifeStage_AnatomicalEntity', 'PhenotypicFeature_LifeStage',
    'Cell_MolecularActivity', 'NamedThing_MolecularEntity', 'Disease_LifeStage',
    'CellularComponent_NamedThing', 'MolecularActivity_NamedThing', 'Protein_NamedThing',
    'ChemicalEntity_NamedThing'
]
before = len(Edges)
Edges = Edges[~Edges['Relation'].isin(relations_to_remove)]
print(f"After removing out-of-scope relations: {len(Edges):,} (removed {before-len(Edges):,})")
print("\nRelation distribution:")
print(Edges['Relation'].value_counts())

After removing out-of-scope relations: 9,036,980 (removed 118,293)

Relation distribution:
Relation
Gene_Gene                             2388156
Gene_AnatomicalEntity                 2033133
Gene_PhenotypicFeature                 782536
Gene_MolecularActivity                 714080
Gene_BiologicalProcess                 655860
                                       ...   
CellularComponent_MolecularEntity           4
Protein_ChemicalEntity                      2
MolecularEntity_ChemicalEntity              2
CellularComponent_Protein                   2
CellularComponent_AnatomicalEntity          1
Name: count, Length: 69, dtype: int64


---
## 4 · Gene_Gene (interacts_with) — per species

Filtered to same-species intra-species interactions only.

In [15]:
#

In [43]:
Gene_Gene_df = Edges[Edges['Relation'] == 'Gene_Gene']
Gene_Gene_df_Interacts = Gene_Gene_df[Gene_Gene_df['Relation_type'] == 'interacts_with'].copy()
Gene_Gene_df_Interacts['Species_Species'] = Gene_Gene_df_Interacts['Head_taxon_name'] + '_' + Gene_Gene_df_Interacts['Tail_taxon_name']

# Keep only intra-species pairs
same_sp = [
    'Homo sapiens_Homo sapiens', 'Mus musculus_Mus musculus',
    'Drosophila melanogaster_Drosophila melanogaster', 'Danio rerio_Danio rerio',
    'Caenorhabditis elegans_Caenorhabditis elegans', 'Rattus norvegicus_Rattus norvegicus'
]
Edges_Final = Gene_Gene_df_Interacts[Gene_Gene_df_Interacts['Species_Species'].isin(same_sp)]
print("Gene_Gene intra-species distribution:")
print(Edges_Final['Species_Species'].value_counts())

Gene_Gene intra-species distribution:
Species_Species
Homo sapiens_Homo sapiens                          1324580
Mus musculus_Mus musculus                           263909
Drosophila melanogaster_Drosophila melanogaster     196222
Rattus norvegicus_Rattus norvegicus                 165110
Danio rerio_Danio rerio                             151490
Caenorhabditis elegans_Caenorhabditis elegans       146558
Name: count, dtype: int64


In [44]:
# ── Human Gene_Gene ───────────────────────────────────────────────────────────
G_G_Human = Edges_Final[Edges_Final['Species_Species'] == 'Homo sapiens_Homo sapiens'].copy()
G_G_Human = resolve_gene_symbol_head(G_G_Human)
G_G_Human = resolve_gene_symbol_tail(G_G_Human)
G_G_Human['Head_ID_IS'] = 'NCBI_ID'
G_G_Human['Tail_ID_IS'] = 'NCBI_ID'
G_G_Human = select_cols(G_G_Human)
save(G_G_Human, os.path.join(OUT_HUMAN, 'GENE_GENE_HUMAN_HUMAN_MONARCH.csv'))

# ── C. elegans Gene_Gene ───────────────────────────────────────────────────────
G_G_Cele = Edges_Final[Edges_Final['Species_Species'] == 'Caenorhabditis elegans_Caenorhabditis elegans'].copy()
G_G_Cele[['Tail_name','Tail']] = G_G_Cele[['Tail','Tail_name']]
G_G_Cele[['Head_name','Head']] = G_G_Cele[['Head','Head_name']]
G_G_Cele['KG_Source'] = 'Monarch_KG'
G_G_Cele = select_cols(G_G_Cele)
save(G_G_Cele, os.path.join(OUT_CELE, 'GENE_GENE_Cele_Cele_MONARCH.csv'))

# ── Drosophila Gene_Gene ────────────────────────────────────────────────────────
G_G_Droso = Edges_Final[Edges_Final['Species_Species'] == 'Drosophila melanogaster_Drosophila melanogaster'].copy()
G_G_Droso[['Tail_name','Tail']] = G_G_Droso[['Tail','Tail_name']]
G_G_Droso[['Head_name','Head']] = G_G_Droso[['Head','Head_name']]
G_G_Droso['KG_Source'] = 'Monarch_KG'
G_G_Droso = select_cols(G_G_Droso)
save(G_G_Droso, os.path.join(OUT_DROSO, 'GENE_GENE_Droso_Droso_MONARCH.csv'))

# ── Zebrafish Gene_Gene ─────────────────────────────────────────────────────────
G_G_Zebra = Edges_Final[Edges_Final['Species_Species'] == 'Danio rerio_Danio rerio'].copy()
G_G_Zebra[['Tail_name','Tail']] = G_G_Zebra[['Tail','Tail_name']]
G_G_Zebra[['Head_name','Head']] = G_G_Zebra[['Head','Head_name']]
G_G_Zebra['KG_Source'] = 'Monarch_KG'
G_G_Zebra = select_cols(G_G_Zebra)
save(G_G_Zebra, os.path.join(OUT_ZEBRA, 'GENE_GENE_Zebrafish_Zebrafish_MONARCH.csv'))

# ── Mouse Gene_Gene ─────────────────────────────────────────────────────────────
G_G_Mouse = Edges_Final[Edges_Final['Species_Species'] == 'Mus musculus_Mus musculus'].copy()
G_G_Mouse[['Tail_name','Tail']] = G_G_Mouse[['Tail','Tail_name']]
G_G_Mouse[['Head_name','Head']] = G_G_Mouse[['Head','Head_name']]
G_G_Mouse['KG_Source'] = 'Monarch_KG'
G_G_Mouse = select_cols(G_G_Mouse)
save(G_G_Mouse, os.path.join(OUT_MOUSE, 'GENE_GENE_Mouse_Mouse_MONARCH.csv'))

# ── Rat Gene_Gene ───────────────────────────────────────────────────────────────
# BUG FIX: original set G_G_Mouse_Interacts_Edges_Final['KG_Source'] here
# instead of G_G_Rat_Interacts_Edges_Final['KG_Source']
G_G_Rat = Edges_Final[Edges_Final['Species_Species'] == 'Rattus norvegicus_Rattus norvegicus'].copy()
G_G_Rat[['Tail_name','Tail']] = G_G_Rat[['Tail','Tail_name']]
G_G_Rat[['Head_name','Head']] = G_G_Rat[['Head','Head_name']]
G_G_Rat['KG_Source'] = 'Monarch_KG'   # BUG FIX: was setting this on Mouse df
G_G_Rat = select_cols(G_G_Rat)
save(G_G_Rat, os.path.join(OUT_RAT, 'GENE_GENE_RAT_RAT_MONARCH.csv'))

Saved 1,324,580 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/GENE_GENE_HUMAN_HUMAN_MONARCH.csv
Saved 146,558 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Celegans/GENE_GENE_Cele_Cele_MONARCH.csv
Saved 196,222 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Drosophila/GENE_GENE_Droso_Droso_MONARCH.csv
Saved 151,490 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Zebrafish/GENE_GENE_Zebrafish_Zebrafish_MONARCH.csv
Saved 263,909 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Mouse/GENE_GENE_Mouse_Mouse_MONARCH.csv
Saved 165,110 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Rat/GENE_GENE_RAT_RAT_MONARCH.csv


---
## 5 · Gene → AnatomicalEntity, BiologicalProcess, PhenotypicFeature, MolecularActivity, CellularComponent

In [45]:
# ── Gene_AnatomicalEntity ────────────────────────────────────────────────────────────────
Gene_AnatomicalEntity_df = Edges[Edges['Relation'] == 'Gene_AnatomicalEntity'].copy()
Gene_AnatomicalEntity_df['Species_Species'] = Gene_AnatomicalEntity_df['Head_taxon_name'] + '_' + Gene_AnatomicalEntity_df['Tail_taxon_name'].fillna(Gene_AnatomicalEntity_df['Head_taxon_name'])

# Homo sapiens
Human_Gene_AnatomicalEntity = Gene_AnatomicalEntity_df[Gene_AnatomicalEntity_df['Head_taxon_name'] == 'Homo sapiens'].copy()
Human_Gene_AnatomicalEntity = resolve_gene_symbol_head(Human_Gene_AnatomicalEntity)
Human_Gene_AnatomicalEntity['Head_ID_IS'] = 'NCBI_ID'
save(select_cols(Human_Gene_AnatomicalEntity), os.path.join(OUT_HUMAN, 'Gene_Human_Anatomy_MONARCH.csv'))

# Saccharomyces cerevisiae S288C
Yeast_Gene_AnatomicalEntity = Gene_AnatomicalEntity_df[Gene_AnatomicalEntity_df['Head_taxon_name'] == 'Saccharomyces cerevisiae S288C'].copy()
save(select_cols(Yeast_Gene_AnatomicalEntity), os.path.join(OUT_YEAST, 'Gene_Yeast_Anatomy_MONARCH.csv'))

# Caenorhabditis elegans
Cele_Gene_AnatomicalEntity = Gene_AnatomicalEntity_df[Gene_AnatomicalEntity_df['Head_taxon_name'] == 'Caenorhabditis elegans'].copy()
save(select_cols(Cele_Gene_AnatomicalEntity), os.path.join(OUT_CELE, 'Gene_Cele_Anatomy_MONARCH.csv'))

# Drosophila melanogaster
Droso_Gene_AnatomicalEntity = Gene_AnatomicalEntity_df[Gene_AnatomicalEntity_df['Head_taxon_name'] == 'Drosophila melanogaster'].copy()
save(select_cols(Droso_Gene_AnatomicalEntity), os.path.join(OUT_DROSO, 'Gene_Droso_Anatomy_MONARCH.csv'))

# Danio rerio
Zebra_Gene_AnatomicalEntity = Gene_AnatomicalEntity_df[Gene_AnatomicalEntity_df['Head_taxon_name'] == 'Danio rerio'].copy()
save(select_cols(Zebra_Gene_AnatomicalEntity), os.path.join(OUT_ZEBRA, 'Gene_Zebra_Anatomy_MONARCH.csv'))


Saved 161,084 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Gene_Human_Anatomy_MONARCH.csv
Saved 0 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Yeast/Gene_Yeast_Anatomy_MONARCH.csv
Saved 105,006 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Celegans/Gene_Cele_Anatomy_MONARCH.csv
Saved 397,421 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Drosophila/Gene_Droso_Anatomy_MONARCH.csv
Saved 528,410 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Zebrafish/Gene_Zebra_Anatomy_MONARCH.csv


In [46]:
# ── Gene_BiologicalProcess ────────────────────────────────────────────────────────────────
Gene_BiologicalProcess_df = Edges[Edges['Relation'] == 'Gene_BiologicalProcess'].copy()
Gene_BiologicalProcess_df['Species_Species'] = Gene_BiologicalProcess_df['Head_taxon_name'] + '_' + Gene_BiologicalProcess_df['Tail_taxon_name'].fillna(Gene_BiologicalProcess_df['Head_taxon_name'])

# Homo sapiens
Human_Gene_BiologicalProcess = Gene_BiologicalProcess_df[Gene_BiologicalProcess_df['Head_taxon_name'] == 'Homo sapiens'].copy()
Human_Gene_BiologicalProcess = resolve_gene_symbol_head(Human_Gene_BiologicalProcess)
Human_Gene_BiologicalProcess['Head_ID_IS'] = 'NCBI_ID'
save(select_cols(Human_Gene_BiologicalProcess), os.path.join(OUT_HUMAN, 'Gene_Human_BiologicalProcess_MONARCH.csv'))

# Saccharomyces cerevisiae S288C
Yeast_Gene_BiologicalProcess = Gene_BiologicalProcess_df[Gene_BiologicalProcess_df['Head_taxon_name'] == 'Saccharomyces cerevisiae S288C'].copy()
save(select_cols(Yeast_Gene_BiologicalProcess), os.path.join(OUT_YEAST, 'Gene_Yeast_BiologicalProcess_MONARCH.csv'))

# Caenorhabditis elegans
Cele_Gene_BiologicalProcess = Gene_BiologicalProcess_df[Gene_BiologicalProcess_df['Head_taxon_name'] == 'Caenorhabditis elegans'].copy()
save(select_cols(Cele_Gene_BiologicalProcess), os.path.join(OUT_CELE, 'Gene_Cele_BiologicalProcess_MONARCH.csv'))

# Drosophila melanogaster
Droso_Gene_BiologicalProcess = Gene_BiologicalProcess_df[Gene_BiologicalProcess_df['Head_taxon_name'] == 'Drosophila melanogaster'].copy()
save(select_cols(Droso_Gene_BiologicalProcess), os.path.join(OUT_DROSO, 'Gene_Droso_BiologicalProcess_MONARCH.csv'))

# Danio rerio
Zebra_Gene_BiologicalProcess = Gene_BiologicalProcess_df[Gene_BiologicalProcess_df['Head_taxon_name'] == 'Danio rerio'].copy()
save(select_cols(Zebra_Gene_BiologicalProcess), os.path.join(OUT_ZEBRA, 'Gene_Zebra_BiologicalProcess_MONARCH.csv'))

# Mus musculus
Mouse_Gene_BiologicalProcess = Gene_BiologicalProcess_df[Gene_BiologicalProcess_df['Head_taxon_name'] == 'Mus musculus'].copy()
save(select_cols(Mouse_Gene_BiologicalProcess), os.path.join(OUT_MOUSE, 'Gene_Mouse_BiologicalProcess_MONARCH.csv'))

# Rattus norvegicus
Rat_Gene_BiologicalProcess = Gene_BiologicalProcess_df[Gene_BiologicalProcess_df['Head_taxon_name'] == 'Rattus norvegicus'].copy()
save(select_cols(Rat_Gene_BiologicalProcess), os.path.join(OUT_RAT, 'Gene_Rat_BiologicalProcess_MONARCH.csv'))


Saved 166,291 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Gene_Human_BiologicalProcess_MONARCH.csv
Saved 38,892 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Yeast/Gene_Yeast_BiologicalProcess_MONARCH.csv
Saved 41,253 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Celegans/Gene_Cele_BiologicalProcess_MONARCH.csv
Saved 59,900 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Drosophila/Gene_Droso_BiologicalProcess_MONARCH.csv
Saved 91,909 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Zebrafish/Gene_Zebra_BiologicalProcess_MONARCH.csv
Saved 257,615 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Mouse/Gene_Mouse_BiologicalProcess_MONARCH.csv
Saved 234,405 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Rat/Gene_Rat_BiologicalProcess_MONARCH.csv


In [47]:
# ── Gene_PhenotypicFeature ────────────────────────────────────────────────────────────────
Gene_PhenotypicFeature_df = Edges[Edges['Relation'] == 'Gene_PhenotypicFeature'].copy()
Gene_PhenotypicFeature_df['Species_Species'] = Gene_PhenotypicFeature_df['Head_taxon_name'] + '_' + Gene_PhenotypicFeature_df['Tail_taxon_name'].fillna(Gene_PhenotypicFeature_df['Head_taxon_name'])

# Homo sapiens
Human_Gene_PhenotypicFeature = Gene_PhenotypicFeature_df[Gene_PhenotypicFeature_df['Head_taxon_name'] == 'Homo sapiens'].copy()
Human_Gene_PhenotypicFeature = resolve_gene_symbol_head(Human_Gene_PhenotypicFeature)
Human_Gene_PhenotypicFeature['Head_ID_IS'] = 'NCBI_ID'
save(select_cols(Human_Gene_PhenotypicFeature), os.path.join(OUT_HUMAN, 'Gene_Human_Phenotype_MONARCH.csv'))

# Saccharomyces cerevisiae S288C
Yeast_Gene_PhenotypicFeature = Gene_PhenotypicFeature_df[Gene_PhenotypicFeature_df['Head_taxon_name'] == 'Saccharomyces cerevisiae S288C'].copy()
save(select_cols(Yeast_Gene_PhenotypicFeature), os.path.join(OUT_YEAST, 'Gene_Yeast_Phenotype_MONARCH.csv'))

# Caenorhabditis elegans
Cele_Gene_PhenotypicFeature = Gene_PhenotypicFeature_df[Gene_PhenotypicFeature_df['Head_taxon_name'] == 'Caenorhabditis elegans'].copy()
save(select_cols(Cele_Gene_PhenotypicFeature), os.path.join(OUT_CELE, 'Gene_Cele_Phenotype_MONARCH.csv'))

# Drosophila melanogaster
Droso_Gene_PhenotypicFeature = Gene_PhenotypicFeature_df[Gene_PhenotypicFeature_df['Head_taxon_name'] == 'Drosophila melanogaster'].copy()
save(select_cols(Droso_Gene_PhenotypicFeature), os.path.join(OUT_DROSO, 'Gene_Droso_Phenotype_MONARCH.csv'))

# Danio rerio
Zebra_Gene_PhenotypicFeature = Gene_PhenotypicFeature_df[Gene_PhenotypicFeature_df['Head_taxon_name'] == 'Danio rerio'].copy()
save(select_cols(Zebra_Gene_PhenotypicFeature), os.path.join(OUT_ZEBRA, 'Gene_Zebra_Phenotype_MONARCH.csv'))

# Mus musculus
Mouse_Gene_PhenotypicFeature = Gene_PhenotypicFeature_df[Gene_PhenotypicFeature_df['Head_taxon_name'] == 'Mus musculus'].copy()
save(select_cols(Mouse_Gene_PhenotypicFeature), os.path.join(OUT_MOUSE, 'Gene_Mouse_Phenotype_MONARCH.csv'))

# Rattus norvegicus
Rat_Gene_PhenotypicFeature = Gene_PhenotypicFeature_df[Gene_PhenotypicFeature_df['Head_taxon_name'] == 'Rattus norvegicus'].copy()
save(select_cols(Rat_Gene_PhenotypicFeature), os.path.join(OUT_RAT, 'Gene_Rat_Phenotype_MONARCH.csv'))


Saved 312,607 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Gene_Human_Phenotype_MONARCH.csv
Saved 0 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Yeast/Gene_Yeast_Phenotype_MONARCH.csv
Saved 26,686 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Celegans/Gene_Cele_Phenotype_MONARCH.csv
Saved 0 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Drosophila/Gene_Droso_Phenotype_MONARCH.csv
Saved 156,719 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Zebrafish/Gene_Zebra_Phenotype_MONARCH.csv
Saved 286,524 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Mouse/Gene_Mouse_Phenotype_MONARCH.csv
Saved 3,434 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Rat/Gene_Rat_Phenotype_MONARCH.csv


In [48]:
# ── Gene_MolecularActivity ────────────────────────────────────────────────────────────────
Gene_MolecularActivity_df = Edges[Edges['Relation'] == 'Gene_MolecularActivity'].copy()
Gene_MolecularActivity_df['Species_Species'] = Gene_MolecularActivity_df['Head_taxon_name'] + '_' + Gene_MolecularActivity_df['Tail_taxon_name'].fillna(Gene_MolecularActivity_df['Head_taxon_name'])

# Homo sapiens
Human_Gene_MolecularActivity = Gene_MolecularActivity_df[Gene_MolecularActivity_df['Head_taxon_name'] == 'Homo sapiens'].copy()
Human_Gene_MolecularActivity = resolve_gene_symbol_head(Human_Gene_MolecularActivity)
Human_Gene_MolecularActivity['Head_ID_IS'] = 'NCBI_ID'
save(select_cols(Human_Gene_MolecularActivity), os.path.join(OUT_HUMAN, 'Gene_Human_MolecularActivity_MONARCH.csv'))

# Saccharomyces cerevisiae S288C
Yeast_Gene_MolecularActivity = Gene_MolecularActivity_df[Gene_MolecularActivity_df['Head_taxon_name'] == 'Saccharomyces cerevisiae S288C'].copy()
save(select_cols(Yeast_Gene_MolecularActivity), os.path.join(OUT_YEAST, 'Gene_Yeast_MolecularActivity_MONARCH.csv'))

# Caenorhabditis elegans
Cele_Gene_MolecularActivity = Gene_MolecularActivity_df[Gene_MolecularActivity_df['Head_taxon_name'] == 'Caenorhabditis elegans'].copy()
save(select_cols(Cele_Gene_MolecularActivity), os.path.join(OUT_CELE, 'Gene_Cele_MolecularActivity_MONARCH.csv'))

# Drosophila melanogaster
Droso_Gene_MolecularActivity = Gene_MolecularActivity_df[Gene_MolecularActivity_df['Head_taxon_name'] == 'Drosophila melanogaster'].copy()
save(select_cols(Droso_Gene_MolecularActivity), os.path.join(OUT_DROSO, 'Gene_Droso_MolecularActivity_MONARCH.csv'))

# Danio rerio
Zebra_Gene_MolecularActivity = Gene_MolecularActivity_df[Gene_MolecularActivity_df['Head_taxon_name'] == 'Danio rerio'].copy()
save(select_cols(Zebra_Gene_MolecularActivity), os.path.join(OUT_ZEBRA, 'Gene_Zebra_MolecularActivity_MONARCH.csv'))

# Mus musculus
Mouse_Gene_MolecularActivity = Gene_MolecularActivity_df[Gene_MolecularActivity_df['Head_taxon_name'] == 'Mus musculus'].copy()
save(select_cols(Mouse_Gene_MolecularActivity), os.path.join(OUT_MOUSE, 'Gene_Mouse_MolecularActivity_MONARCH.csv'))

# Rattus norvegicus
Rat_Gene_MolecularActivity = Gene_MolecularActivity_df[Gene_MolecularActivity_df['Head_taxon_name'] == 'Rattus norvegicus'].copy()
save(select_cols(Rat_Gene_MolecularActivity), os.path.join(OUT_RAT, 'Gene_Rat_MolecularActivity_MONARCH.csv'))


Saved 347,707 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Gene_Human_MolecularActivity_MONARCH.csv
Saved 37,189 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Yeast/Gene_Yeast_MolecularActivity_MONARCH.csv
Saved 39,781 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Celegans/Gene_Cele_MolecularActivity_MONARCH.csv
Saved 38,735 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Drosophila/Gene_Droso_MolecularActivity_MONARCH.csv
Saved 87,998 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Zebrafish/Gene_Zebra_MolecularActivity_MONARCH.csv
Saved 162,670 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Mouse/Gene_Mouse_MolecularActivity_MONARCH.csv
Saved 114,015 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Rat/Gene_Rat_MolecularActivity_MONARCH.csv


In [49]:
# ── Gene_CellularComponent ────────────────────────────────────────────────────────────────
Gene_CellularComponent_df = Edges[Edges['Relation'] == 'Gene_CellularComponent'].copy()
Gene_CellularComponent_df['Species_Species'] = Gene_CellularComponent_df['Head_taxon_name'] + '_' + Gene_CellularComponent_df['Tail_taxon_name'].fillna(Gene_CellularComponent_df['Head_taxon_name'])

# Homo sapiens
Human_Gene_CellularComponent = Gene_CellularComponent_df[Gene_CellularComponent_df['Head_taxon_name'] == 'Homo sapiens'].copy()
Human_Gene_CellularComponent = resolve_gene_symbol_head(Human_Gene_CellularComponent)
Human_Gene_CellularComponent['Head_ID_IS'] = 'NCBI_ID'
save(select_cols(Human_Gene_CellularComponent), os.path.join(OUT_HUMAN, 'Gene_Human_CellularComponent_MONARCH.csv'))

# Saccharomyces cerevisiae S288C
Yeast_Gene_CellularComponent = Gene_CellularComponent_df[Gene_CellularComponent_df['Head_taxon_name'] == 'Saccharomyces cerevisiae S288C'].copy()
save(select_cols(Yeast_Gene_CellularComponent), os.path.join(OUT_YEAST, 'Gene_Yeast_CellularComponent_MONARCH.csv'))

# Caenorhabditis elegans
Cele_Gene_CellularComponent = Gene_CellularComponent_df[Gene_CellularComponent_df['Head_taxon_name'] == 'Caenorhabditis elegans'].copy()
save(select_cols(Cele_Gene_CellularComponent), os.path.join(OUT_CELE, 'Gene_Cele_CellularComponent_MONARCH.csv'))

# Drosophila melanogaster
Droso_Gene_CellularComponent = Gene_CellularComponent_df[Gene_CellularComponent_df['Head_taxon_name'] == 'Drosophila melanogaster'].copy()
save(select_cols(Droso_Gene_CellularComponent), os.path.join(OUT_DROSO, 'Gene_Droso_CellularComponent_MONARCH.csv'))

# Danio rerio
Zebra_Gene_CellularComponent = Gene_CellularComponent_df[Gene_CellularComponent_df['Head_taxon_name'] == 'Danio rerio'].copy()
save(select_cols(Zebra_Gene_CellularComponent), os.path.join(OUT_ZEBRA, 'Gene_Zebra_CellularComponent_MONARCH.csv'))

# Mus musculus
Mouse_Gene_CellularComponent = Gene_CellularComponent_df[Gene_CellularComponent_df['Head_taxon_name'] == 'Mus musculus'].copy()
save(select_cols(Mouse_Gene_CellularComponent), os.path.join(OUT_MOUSE, 'Gene_Mouse_CellularComponent_MONARCH.csv'))

# Rattus norvegicus
Rat_Gene_CellularComponent = Gene_CellularComponent_df[Gene_CellularComponent_df['Head_taxon_name'] == 'Rattus norvegicus'].copy()
save(select_cols(Rat_Gene_CellularComponent), os.path.join(OUT_RAT, 'Gene_Rat_CellularComponent_MONARCH.csv'))


Saved 201,613 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Gene_Human_CellularComponent_MONARCH.csv
Saved 60,088 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Yeast/Gene_Yeast_CellularComponent_MONARCH.csv
Saved 40,938 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Celegans/Gene_Cele_CellularComponent_MONARCH.csv
Saved 50,087 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Drosophila/Gene_Droso_CellularComponent_MONARCH.csv
Saved 67,369 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Zebrafish/Gene_Zebra_CellularComponent_MONARCH.csv
Saved 174,588 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Mouse/Gene_Mouse_CellularComponent_MONARCH.csv
Saved 139,567 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Rat/Gene_Rat_CellularComponent_MONARCH.csv


---
## 6 · PhenotypicFeature_PhenotypicFeature and Disease_PhenotypicFeature

In [50]:
# ── PhenotypicFeature_PhenotypicFeature ────────────────────────────────────────
# Note: no species annotation in this relation — treated as cross-species or Human
PhenoFeat_PhenoFeat = Edges[Edges['Relation'] == 'PhenotypicFeature_PhenotypicFeature'].copy()
PhenoFeat_PhenoFeat['Species_Species'] = (PhenoFeat_PhenoFeat['Head_taxon_name'] + '_' +
    PhenoFeat_PhenoFeat['Tail_taxon_name'].fillna(PhenoFeat_PhenoFeat['Head_taxon_name']))
print(f"PhenotypicFeature_PhenotypicFeature: {len(PhenoFeat_PhenoFeat):,} rows")
# Not exported — no species definition available; kept for inspection

PhenotypicFeature_PhenotypicFeature: 264,410 rows


/tmp/ipykernel_1542094/269584969.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  PhenoFeat_PhenoFeat['Tail_taxon_name'].fillna(PhenoFeat_PhenoFeat['Head_taxon_name']))


In [51]:
# ── Disease_PhenotypicFeature ──────────────────────────────────────────────────
# No species annotation; NBO Tail IDs removed (not a target ontology)
Disease_PhenotypicFeature = Edges[Edges['Relation'] == 'Disease_PhenotypicFeature'].copy()
Disease_PhenotypicFeature['Species_Species'] = (Disease_PhenotypicFeature['Head_taxon_name'] + '_' +
    Disease_PhenotypicFeature['Tail_taxon_name'].fillna(Disease_PhenotypicFeature['Head_taxon_name']))
Disease_PhenotypicFeature = Disease_PhenotypicFeature[Disease_PhenotypicFeature['Tail_ID_IS'] != 'NBO']
Disease_PhenotypicFeature['Head_DO_name'] = (
    Disease_PhenotypicFeature['Head_name'].map(DOID_disname_dict)
    .fillna(Disease_PhenotypicFeature['Head_name'].map(MONDO_dict))
    .fillna(Disease_PhenotypicFeature['Head']))
Disease_PhenotypicFeature[['Head','Head_DO_name']] = Disease_PhenotypicFeature[['Head_DO_name','Head']]
Disease_PhenotypicFeature['Head_ID_IS'] = np.where(
    Disease_PhenotypicFeature['Head'].isna(), None,
    np.where(Disease_PhenotypicFeature['Head'].str.startswith('DOID'), 'DOID', 'MESH'))
Disease_PhenotypicFeature['Tail_ID_IS'] = 'HPO_ID'
Disease_PhenotypicFeature['KG_Source']  = 'Monarch_KG'
Disease_PhenotypicFeature['Relation']   = 'Disease_Phenotype'
Disease_PhenotypicFeature['Tail_type']  = 'Phenotype'
Disease_PhenotypicFeature = Disease_PhenotypicFeature.rename(columns={'Head_DO_name': 'Head_Orignal'})
Disease_PhenotypicFeature = select_cols(Disease_PhenotypicFeature)
save(Disease_PhenotypicFeature, os.path.join(OUT_HUMAN, 'Disease_Phenotype_human_MONARCH.csv'))

/tmp/ipykernel_1542094/1132665150.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Disease_PhenotypicFeature['Tail_taxon_name'].fillna(Disease_PhenotypicFeature['Head_taxon_name']))


Saved 260,707 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Disease_Phenotype_human_MONARCH.csv


---
## 7 · AnatomicalEntity_AnatomicalEntity

Species inferred from ontology prefix (UBERON=Human, FBbt=Droso, WBbt=Cele, ZFA=Zebrafish, EMAPA=Mouse).

In [52]:
AnatomicalEntity_AnatomicalEntity = Edges[Edges['Relation'] == 'AnatomicalEntity_AnatomicalEntity'].copy()
AnatomicalEntity_AnatomicalEntity['Species_Species'] = (AnatomicalEntity_AnatomicalEntity['Head_taxon_name'] + '_' +
    AnatomicalEntity_AnatomicalEntity['Tail_taxon_name'].fillna(AnatomicalEntity_AnatomicalEntity['Head_taxon_name']))

Rel_Source = {'FBbt':'Drosophila melanogaster','UBERON':'Homo sapiens','WBbt':'Caenorhabditis elegans',
              'ZFA':'Danio rerio','EMAPA':'Mus musculus'}
AnatomicalEntity_AnatomicalEntity['Head_taxon_name'] = AnatomicalEntity_AnatomicalEntity['Head_ID_IS'].map(Rel_Source).fillna(np.nan)
AnatomicalEntity_AnatomicalEntity['Tail_taxon_name'] = AnatomicalEntity_AnatomicalEntity['Tail_ID_IS'].map(Rel_Source).fillna(np.nan)
AnatomicalEntity_AnatomicalEntity = AnatomicalEntity_AnatomicalEntity[~AnatomicalEntity_AnatomicalEntity['Head_taxon_name'].isna()]
AnatomicalEntity_AnatomicalEntity = AnatomicalEntity_AnatomicalEntity[~AnatomicalEntity_AnatomicalEntity['Tail_taxon_name'].isna()]
AnatomicalEntity_AnatomicalEntity['Species_Species'] = (AnatomicalEntity_AnatomicalEntity['Head_taxon_name'] + '_' +
    AnatomicalEntity_AnatomicalEntity['Tail_taxon_name'])

for sp, var, out in [('Homo sapiens','Hum','HUMAN'),('Caenorhabditis elegans','Cele','CELE'),
                     ('Drosophila melanogaster','Droso','DROSO'),('Danio rerio','Zebra','ZEBRA'),
                     ('Mus musculus','Mouse','MOUSE')]:
    sub = AnatomicalEntity_AnatomicalEntity[AnatomicalEntity_AnatomicalEntity['Species_Species'] == f'{sp}_{sp}']
    save(sub, os.path.join(eval(f'OUT_{out}'), f'{var}_AnatomicalEntity_AnatomicalEntity_MONARCH.csv'))

/tmp/ipykernel_1542094/992695489.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  AnatomicalEntity_AnatomicalEntity['Tail_taxon_name'].fillna(AnatomicalEntity_AnatomicalEntity['Head_taxon_name']))


Saved 36,890 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Hum_AnatomicalEntity_AnatomicalEntity_MONARCH.csv
Saved 15,092 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Celegans/Cele_AnatomicalEntity_AnatomicalEntity_MONARCH.csv
Saved 131,953 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Drosophila/Droso_AnatomicalEntity_AnatomicalEntity_MONARCH.csv
Saved 5,951 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Zebrafish/Zebra_AnatomicalEntity_AnatomicalEntity_MONARCH.csv
Saved 4,579 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Mouse/Mouse_AnatomicalEntity_AnatomicalEntity_MONARCH.csv


---
## 8 · Gene_Pathway — per species

Tail Reactome IDs normalised to R-HSA prefix.

In [53]:
Gene_Pathway = Edges[Edges['Relation'] == 'Gene_Pathway'].copy()
Gene_Pathway['Tail'] = Gene_Pathway['Tail'].str.replace('Reactome:', '', regex=False)
Gene_Pathway['Tail'] = Gene_Pathway['Tail'].str.replace(r'R-[A-Z]{3}', 'R-HSA', regex=True)
Gene_Pathway = Gene_Pathway.drop_duplicates(subset=['Head','Relation','Tail'])
Gene_Pathway['Tail_taxon_name'] = 'Homo sapiens'
Gene_Pathway['Species_Species'] = Gene_Pathway['Head_taxon_name'] + '_Homo sapiens'
Gene_Pathway = Gene_Pathway[Gene_Pathway['Species_Species'] != 'Mus musculus_Homo sapiens']

Human_Gene_Pathway = Gene_Pathway[Gene_Pathway['Head_taxon_name'] == 'Homo sapiens'].copy()
Human_Gene_Pathway = resolve_gene_symbol_head(Human_Gene_Pathway)
Human_Gene_Pathway = select_cols(Human_Gene_Pathway)
save(Human_Gene_Pathway, os.path.join(OUT_HUMAN, 'Human_Gene_Pathway_human_MONARCH.csv'))

for sp, var, out in [('Caenorhabditis elegans','Cele','CELE'),
                     ('Drosophila melanogaster','Droso','DROSO'),
                     ('Danio rerio','Zebra','ZEBRA'),
                     ('Rattus norvegicus','Rat','RAT')]:
    sub = Gene_Pathway[Gene_Pathway['Head_taxon_name'] == sp]
    save(sub, os.path.join(eval(f'OUT_{out}'), f'{var}_Gene_Pathway_MONARCH.csv'))

Saved 39,128 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Human_Gene_Pathway_human_MONARCH.csv
Saved 12,698 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Celegans/Cele_Gene_Pathway_MONARCH.csv
Saved 18,469 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Drosophila/Droso_Gene_Pathway_MONARCH.csv
Saved 10,398 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Zebrafish/Zebra_Gene_Pathway_MONARCH.csv
Saved 20,026 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Rat/Rat_Gene_Pathway_MONARCH.csv


---
## 9 · PhenotypicFeature_AnatomicalEntity

Species inferred from Head ID prefix. C. elegans cross-species rows excluded.

In [54]:
PhenotypicFeature_AnatomicalEntity = Edges[Edges['Relation'] == 'PhenotypicFeature_AnatomicalEntity'].copy()
PhenotypicFeature_AnatomicalEntity['Species_Species'] = (PhenotypicFeature_AnatomicalEntity['Head_taxon_name'] + '_' +
    PhenotypicFeature_AnatomicalEntity['Tail_taxon_name'].fillna(PhenotypicFeature_AnatomicalEntity['Head_taxon_name']))
PhenotypicFeature_AnatomicalEntity = PhenotypicFeature_AnatomicalEntity[
    PhenotypicFeature_AnatomicalEntity['Species_Species'] != 'Caenorhabditis elegans_Homo sapiens']

Head_Rel_Source = {'WBPhenotype':'Caenorhabditis elegans','HP':'Homo sapiens','UPHENO':'Homo sapiens',
                   'ZFA':'Danio rerio','MP':'Mus musculus'}
# Not further processed — available for downstream use
print(f"PhenotypicFeature_AnatomicalEntity: {len(PhenotypicFeature_AnatomicalEntity):,} rows")
print("Head species inferred from ID prefix:", Head_Rel_Source)

PhenotypicFeature_AnatomicalEntity: 107,201 rows
Head species inferred from ID prefix: {'WBPhenotype': 'Caenorhabditis elegans', 'HP': 'Homo sapiens', 'UPHENO': 'Homo sapiens', 'ZFA': 'Danio rerio', 'MP': 'Mus musculus'}


/tmp/ipykernel_1542094/1480984788.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  PhenotypicFeature_AnatomicalEntity['Tail_taxon_name'].fillna(PhenotypicFeature_AnatomicalEntity['Head_taxon_name']))


---
## 10 · ChemicalEntity_Pathway (F2)

Head: ChEBI → PubChem CID. Tail: Reactome pathway.

In [55]:
ChemicalEntity_Pathway = Edges[Edges['Relation'] == 'ChemicalEntity_Pathway'].copy()
ChemicalEntity_Pathway['Tail'] = ChemicalEntity_Pathway['Tail'].str.replace('Reactome:', '', regex=False)
ChemicalEntity_Pathway['Tail'] = ChemicalEntity_Pathway['Tail'].str.replace(r'R-[A-Z]{3}', 'R-HSA', regex=True)
ChemicalEntity_Pathway = ChemicalEntity_Pathway.drop_duplicates(subset=['Head','Relation','Tail'])
ChemicalEntity_Pathway = resolve_chebi_to_pubchem(ChemicalEntity_Pathway, 'Head')
ChemicalEntity_Pathway['Head_taxon_name'] = 'Homo sapiens'
ChemicalEntity_Pathway['Tail_taxon_name'] = 'Homo sapiens'
ChemicalEntity_Pathway['Head_ID_IS'] = 'Pubchem'
ChemicalEntity_Pathway['Species_Species'] = 'Homo sapiens_Homo sapiens'
save(ChemicalEntity_Pathway, os.path.join(OUT_HUMAN, 'Human_Chemical_Pathway_Monarch.csv'))

  ChEBI Head -> PubChem: 8,668 kept / 2,291 dropped
Saved 8,668 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Human_Chemical_Pathway_Monarch.csv


---
## 11 · BiologicalProcess_BiologicalProcess (F2)

In [56]:
BiologicalProcess_BiologicalProcess = Edges[Edges['Relation'] == 'BiologicalProcess_BiologicalProcess'].copy()
BiologicalProcess_BiologicalProcess['Species_Species'] = (BiologicalProcess_BiologicalProcess['Head_taxon_name'] + '_' +
    BiologicalProcess_BiologicalProcess['Tail_taxon_name'].fillna(BiologicalProcess_BiologicalProcess['Head_taxon_name']))
BiologicalProcess_BiologicalProcess = BiologicalProcess_BiologicalProcess.drop_duplicates(subset=['Head','Relation','Tail'])
save(BiologicalProcess_BiologicalProcess, os.path.join(OUT_HUMAN, 'Human_BiologicalProcess_BiologicalProcess_Monarch.csv'))

/tmp/ipykernel_1542094/3024093743.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  BiologicalProcess_BiologicalProcess['Tail_taxon_name'].fillna(BiologicalProcess_BiologicalProcess['Head_taxon_name']))


Saved 53,115 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Human_BiologicalProcess_BiologicalProcess_Monarch.csv


---
## 12 · ChemicalEntity_ChemicalEntity (F2)

**BUG FIX:** original used `BiologicalProcess_BiologicalProcess['Tail_taxon_name']` for Species_Species.

In [57]:
ChemicalEntity_ChemicalEntity = Edges[Edges['Relation'] == 'ChemicalEntity_ChemicalEntity'].copy()
# BUG FIX: original cross-referenced BiologicalProcess_BiologicalProcess df here
ChemicalEntity_ChemicalEntity['Species_Species'] = (ChemicalEntity_ChemicalEntity['Head_taxon_name'] + '_' +
    ChemicalEntity_ChemicalEntity['Tail_taxon_name'].fillna(ChemicalEntity_ChemicalEntity['Head_taxon_name']))
ChemicalEntity_ChemicalEntity = ChemicalEntity_ChemicalEntity.drop_duplicates(subset=['Head','Relation','Tail'])
ChemicalEntity_ChemicalEntity = resolve_chebi_to_pubchem(ChemicalEntity_ChemicalEntity, 'Head')
ChemicalEntity_ChemicalEntity = resolve_chebi_to_pubchem(ChemicalEntity_ChemicalEntity, 'Tail')
ChemicalEntity_ChemicalEntity['Head_taxon_name'] = 'Homo sapiens'
ChemicalEntity_ChemicalEntity['Tail_taxon_name'] = 'Homo sapiens'
ChemicalEntity_ChemicalEntity['Head_ID_IS'] = 'Pubchem'
ChemicalEntity_ChemicalEntity['Tail_ID_IS'] = 'Pubchem'
save(ChemicalEntity_ChemicalEntity, os.path.join(OUT_HUMAN, 'Human_Chemical_Chemical_Monarch.csv'))

/tmp/ipykernel_1542094/3972068391.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ChemicalEntity_ChemicalEntity['Tail_taxon_name'].fillna(ChemicalEntity_ChemicalEntity['Head_taxon_name']))


  ChEBI Head -> PubChem: 32,870 kept / 14,401 dropped
  ChEBI Tail -> PubChem: 11,407 kept / 21,463 dropped
Saved 11,407 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Human_Chemical_Chemical_Monarch.csv


---
## 13 · Disease_Disease (F2)

In [58]:
Disease_Disease = Edges[Edges['Relation'] == 'Disease_Disease'].copy()
Disease_Disease['Species_Species'] = (Disease_Disease['Head_taxon_name'] + '_' +
    Disease_Disease['Tail_taxon_name'].fillna(Disease_Disease['Head_taxon_name']))
Disease_Disease = Disease_Disease.drop_duplicates(subset=['Head','Relation','Tail'])
Disease_Disease = Disease_Disease[~(Disease_Disease['Tail_ID_IS'] == 'MPATH')]
Disease_Disease['Head_DO_name'] = (Disease_Disease['Head_name'].map(DOID_disname_dict)
    .fillna(Disease_Disease['Head_name'].map(MONDO_dict)).fillna(Disease_Disease['Head']))
Disease_Disease['Tail_DO_name'] = (Disease_Disease['Tail_name'].map(DOID_disname_dict)
    .fillna(Disease_Disease['Tail_name'].map(MONDO_dict)).fillna(Disease_Disease['Tail']))
Disease_Disease[['Head','Head_DO_name']] = Disease_Disease[['Head_DO_name','Head']]
Disease_Disease[['Tail','Tail_DO_name']] = Disease_Disease[['Tail_DO_name','Tail']]
Disease_Disease['Head_ID_IS'] = np.where(Disease_Disease['Head'].isna(), None,
    np.where(Disease_Disease['Head'].str.startswith('DOID'), 'DOID', 'MONDO'))
Disease_Disease['Tail_ID_IS'] = np.where(Disease_Disease['Tail'].isna(), None,
    np.where(Disease_Disease['Tail'].str.startswith('DOID'), 'DOID', 'MONDO'))
Disease_Disease['Head_taxon_name'] = 'Homo sapiens'
Disease_Disease['Tail_taxon_name'] = 'Homo sapiens'
save(Disease_Disease, os.path.join(OUT_HUMAN, 'Human_Disease_Disease_Monarch.csv'))

/tmp/ipykernel_1542094/3771520404.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Disease_Disease['Tail_taxon_name'].fillna(Disease_Disease['Head_taxon_name']))


Saved 39,850 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Human_Disease_Disease_Monarch.csv


---
## 14 · Protein_Protein (F2)

In [59]:
Protein_Protein = Edges[Edges['Relation'] == 'Protein_Protein'].copy()
Protein_Protein['Species_Species'] = (Protein_Protein['Head_taxon_name'] + '_' +
    Protein_Protein['Tail_taxon_name'].fillna(Protein_Protein['Head_taxon_name']))
Protein_Protein = Protein_Protein.drop_duplicates(subset=['Head','Relation','Tail'])
Protein_Protein['Protein_Uniprot_tail'] = Protein_Protein['Tail_name'].map(Uniprot_Recname_dict)
Protein_Protein['Protein_Uniprot_head'] = Protein_Protein['Head_name'].map(Uniprot_Recname_dict)
Protein_Protein = Protein_Protein[~Protein_Protein['Protein_Uniprot_head'].isna()]
Protein_Protein = Protein_Protein[~Protein_Protein['Protein_Uniprot_tail'].isna()]
save(Protein_Protein, os.path.join(OUT_HUMAN, 'Human_Protein_Protein_Monarch.csv'))

Saved 18 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Human_Protein_Protein_Monarch.csv


/tmp/ipykernel_1542094/2177176384.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Protein_Protein['Tail_taxon_name'].fillna(Protein_Protein['Head_taxon_name']))


---
## 15 · PhenotypicFeature → BiologicalProcess / CellularComponent / ChemicalEntity / MolecularActivity (F2)

In [60]:
# ── PhenotypicFeature_BiologicalProcess ─────────────────────────────────────────────────────────
PhenotypicFeature_BiologicalProcess = Edges[Edges['Relation'] == 'PhenotypicFeature_BiologicalProcess'].copy()
PhenotypicFeature_BiologicalProcess['Species_Species'] = (PhenotypicFeature_BiologicalProcess['Head_taxon_name'] + '_' + PhenotypicFeature_BiologicalProcess['Tail_taxon_name'].fillna(PhenotypicFeature_BiologicalProcess['Head_taxon_name']))
PhenotypicFeature_BiologicalProcess = PhenotypicFeature_BiologicalProcess.drop_duplicates(subset=['Head','Relation','Tail'])
PhenotypicFeature_BiologicalProcess = PhenotypicFeature_BiologicalProcess[PhenotypicFeature_BiologicalProcess['Head_ID_IS'].isin(['ZP','UPHENO','MP','WBPhenotype','HP'])]
Head_Rel_Source = {'WBPhenotype':'Caenorhabditis elegans','HP':'Homo sapiens','UPHENO':'Homo sapiens','ZP':'Danio rerio','MP':'Mus musculus'}
PhenotypicFeature_BiologicalProcess['Head_taxon_name'] = PhenotypicFeature_BiologicalProcess['Head_ID_IS'].map(Head_Rel_Source).fillna(np.nan)
PhenotypicFeature_BiologicalProcess = PhenotypicFeature_BiologicalProcess[~PhenotypicFeature_BiologicalProcess['Head_taxon_name'].isna()]
sub = PhenotypicFeature_BiologicalProcess[PhenotypicFeature_BiologicalProcess['Head_taxon_name'] == 'Homo sapiens']
save(sub, os.path.join(OUT_HUMAN, 'Human_PhenotypicFeature_BiologicalProcess_Monarch.csv'))
sub = PhenotypicFeature_BiologicalProcess[PhenotypicFeature_BiologicalProcess['Head_taxon_name'] == 'Caenorhabditis elegans']
save(sub, os.path.join(OUT_CELE, 'Cele_PhenotypicFeature_BiologicalProcess_Monarch.csv'))
sub = PhenotypicFeature_BiologicalProcess[PhenotypicFeature_BiologicalProcess['Head_taxon_name'] == 'Danio rerio']
save(sub, os.path.join(OUT_ZEBRA, 'Zebra_PhenotypicFeature_BiologicalProcess_Monarch.csv'))
sub = PhenotypicFeature_BiologicalProcess[PhenotypicFeature_BiologicalProcess['Head_taxon_name'] == 'Mus musculus']
save(sub, os.path.join(OUT_MOUSE, 'Mouse_PhenotypicFeature_BiologicalProcess_Monarch.csv'))

Saved 8,499 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Human_PhenotypicFeature_BiologicalProcess_Monarch.csv
Saved 488 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Celegans/Cele_PhenotypicFeature_BiologicalProcess_Monarch.csv
Saved 10,674 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Zebrafish/Zebra_PhenotypicFeature_BiologicalProcess_Monarch.csv
Saved 1,493 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Mouse/Mouse_PhenotypicFeature_BiologicalProcess_Monarch.csv


/tmp/ipykernel_1542094/2329838682.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  PhenotypicFeature_BiologicalProcess['Species_Species'] = (PhenotypicFeature_BiologicalProcess['Head_taxon_name'] + '_' + PhenotypicFeature_BiologicalProcess['Tail_taxon_name'].fillna(PhenotypicFeature_BiologicalProcess['Head_taxon_name']))


In [61]:
# ── PhenotypicFeature_CellularComponent ─────────────────────────────────────────────────────────
PhenotypicFeature_CellularComponent = Edges[Edges['Relation'] == 'PhenotypicFeature_CellularComponent'].copy()
PhenotypicFeature_CellularComponent['Species_Species'] = (PhenotypicFeature_CellularComponent['Head_taxon_name'] + '_' + PhenotypicFeature_CellularComponent['Tail_taxon_name'].fillna(PhenotypicFeature_CellularComponent['Head_taxon_name']))
PhenotypicFeature_CellularComponent = PhenotypicFeature_CellularComponent.drop_duplicates(subset=['Head','Relation','Tail'])
PhenotypicFeature_CellularComponent = PhenotypicFeature_CellularComponent[PhenotypicFeature_CellularComponent['Head_ID_IS'].isin(['ZP','UPHENO','MP','WBPhenotype','HP'])]
Head_Rel_Source = {'WBPhenotype':'Caenorhabditis elegans','HP':'Homo sapiens','UPHENO':'Homo sapiens','ZP':'Danio rerio','MP':'Mus musculus'}
PhenotypicFeature_CellularComponent['Head_taxon_name'] = PhenotypicFeature_CellularComponent['Head_ID_IS'].map(Head_Rel_Source).fillna(np.nan)
PhenotypicFeature_CellularComponent = PhenotypicFeature_CellularComponent[~PhenotypicFeature_CellularComponent['Head_taxon_name'].isna()]
sub = PhenotypicFeature_CellularComponent[PhenotypicFeature_CellularComponent['Head_taxon_name'] == 'Homo sapiens']
save(sub, os.path.join(OUT_HUMAN, 'Human_Phenotype_CellularComponent_Monarch.csv'))
sub = PhenotypicFeature_CellularComponent[PhenotypicFeature_CellularComponent['Head_taxon_name'] == 'Caenorhabditis elegans']
save(sub, os.path.join(OUT_CELE, 'Cele_Phenotype_CellularComponent_Monarch.csv'))
sub = PhenotypicFeature_CellularComponent[PhenotypicFeature_CellularComponent['Head_taxon_name'] == 'Danio rerio']
save(sub, os.path.join(OUT_ZEBRA, 'Zebra_Phenotype_CellularComponent_Monarch.csv'))
sub = PhenotypicFeature_CellularComponent[PhenotypicFeature_CellularComponent['Head_taxon_name'] == 'Mus musculus']
save(sub, os.path.join(OUT_MOUSE, 'Mouse_Phenotype_CellularComponent_Monarch.csv'))

Saved 581 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Human_Phenotype_CellularComponent_Monarch.csv
Saved 76 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Celegans/Cele_Phenotype_CellularComponent_Monarch.csv
Saved 5,378 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Zebrafish/Zebra_Phenotype_CellularComponent_Monarch.csv
Saved 354 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Mouse/Mouse_Phenotype_CellularComponent_Monarch.csv


/tmp/ipykernel_1542094/2360487170.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  PhenotypicFeature_CellularComponent['Species_Species'] = (PhenotypicFeature_CellularComponent['Head_taxon_name'] + '_' + PhenotypicFeature_CellularComponent['Tail_taxon_name'].fillna(PhenotypicFeature_CellularComponent['Head_taxon_name']))


In [62]:
# ── PhenotypicFeature_ChemicalEntity ─────────────────────────────────────────────────────────
PhenotypicFeature_ChemicalEntity = Edges[Edges['Relation'] == 'PhenotypicFeature_ChemicalEntity'].copy()
PhenotypicFeature_ChemicalEntity['Species_Species'] = (PhenotypicFeature_ChemicalEntity['Head_taxon_name'] + '_' + PhenotypicFeature_ChemicalEntity['Tail_taxon_name'].fillna(PhenotypicFeature_ChemicalEntity['Head_taxon_name']))
PhenotypicFeature_ChemicalEntity = PhenotypicFeature_ChemicalEntity.drop_duplicates(subset=['Head','Relation','Tail'])
PhenotypicFeature_ChemicalEntity = PhenotypicFeature_ChemicalEntity[PhenotypicFeature_ChemicalEntity['Head_ID_IS'].isin(['ZP','UPHENO','MP','WBPhenotype','HP'])]
Head_Rel_Source = {'WBPhenotype':'Caenorhabditis elegans','HP':'Homo sapiens','UPHENO':'Homo sapiens','ZP':'Danio rerio','MP':'Mus musculus'}
PhenotypicFeature_ChemicalEntity['Head_taxon_name'] = PhenotypicFeature_ChemicalEntity['Head_ID_IS'].map(Head_Rel_Source).fillna(np.nan)
PhenotypicFeature_ChemicalEntity = PhenotypicFeature_ChemicalEntity[~PhenotypicFeature_ChemicalEntity['Head_taxon_name'].isna()]
PhenotypicFeature_ChemicalEntity = resolve_chebi_to_pubchem(PhenotypicFeature_ChemicalEntity, 'Tail')
sub = PhenotypicFeature_ChemicalEntity[PhenotypicFeature_ChemicalEntity['Head_taxon_name'] == 'Homo sapiens']
save(sub, os.path.join(OUT_HUMAN, 'Human_Phenotype_ChemicalEntity_Monarch.csv'))
sub = PhenotypicFeature_ChemicalEntity[PhenotypicFeature_ChemicalEntity['Head_taxon_name'] == 'Caenorhabditis elegans']
save(sub, os.path.join(OUT_CELE, 'Cele_Phenotype_ChemicalEntity_Monarch.csv'))
sub = PhenotypicFeature_ChemicalEntity[PhenotypicFeature_ChemicalEntity['Head_taxon_name'] == 'Danio rerio']
save(sub, os.path.join(OUT_ZEBRA, 'Zebra_Phenotype_ChemicalEntity_Monarch.csv'))
sub = PhenotypicFeature_ChemicalEntity[PhenotypicFeature_ChemicalEntity['Head_taxon_name'] == 'Mus musculus']
save(sub, os.path.join(OUT_MOUSE, 'Mouse_Phenotype_ChemicalEntity_Monarch.csv'))

  ChEBI Tail -> PubChem: 2,279 kept / 2,016 dropped
Saved 1,115 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Human_Phenotype_ChemicalEntity_Monarch.csv
Saved 11 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Celegans/Cele_Phenotype_ChemicalEntity_Monarch.csv
Saved 702 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Zebrafish/Zebra_Phenotype_ChemicalEntity_Monarch.csv
Saved 451 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Mouse/Mouse_Phenotype_ChemicalEntity_Monarch.csv


/tmp/ipykernel_1542094/4150684019.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  PhenotypicFeature_ChemicalEntity['Species_Species'] = (PhenotypicFeature_ChemicalEntity['Head_taxon_name'] + '_' + PhenotypicFeature_ChemicalEntity['Tail_taxon_name'].fillna(PhenotypicFeature_ChemicalEntity['Head_taxon_name']))


In [63]:
# ── PhenotypicFeature_MolecularActivity ─────────────────────────────────────────────────────────
PhenotypicFeature_MolecularActivity = Edges[Edges['Relation'] == 'PhenotypicFeature_MolecularActivity'].copy()
PhenotypicFeature_MolecularActivity['Species_Species'] = (PhenotypicFeature_MolecularActivity['Head_taxon_name'] + '_' + PhenotypicFeature_MolecularActivity['Tail_taxon_name'].fillna(PhenotypicFeature_MolecularActivity['Head_taxon_name']))
PhenotypicFeature_MolecularActivity = PhenotypicFeature_MolecularActivity.drop_duplicates(subset=['Head','Relation','Tail'])
PhenotypicFeature_MolecularActivity = PhenotypicFeature_MolecularActivity[PhenotypicFeature_MolecularActivity['Head_ID_IS'].isin(['ZP','UPHENO','MP','WBPhenotype','HP'])]
Head_Rel_Source = {'WBPhenotype':'Caenorhabditis elegans','HP':'Homo sapiens','UPHENO':'Homo sapiens','ZP':'Danio rerio','MP':'Mus musculus'}
PhenotypicFeature_MolecularActivity['Head_taxon_name'] = PhenotypicFeature_MolecularActivity['Head_ID_IS'].map(Head_Rel_Source).fillna(np.nan)
PhenotypicFeature_MolecularActivity = PhenotypicFeature_MolecularActivity[~PhenotypicFeature_MolecularActivity['Head_taxon_name'].isna()]
sub = PhenotypicFeature_MolecularActivity[PhenotypicFeature_MolecularActivity['Head_taxon_name'] == 'Homo sapiens']
save(sub, os.path.join(OUT_HUMAN, 'Human_PhenotypicFeature_MolecularActivity_Monarch.csv'))
sub = PhenotypicFeature_MolecularActivity[PhenotypicFeature_MolecularActivity['Head_taxon_name'] == 'Caenorhabditis elegans']
save(sub, os.path.join(OUT_CELE, 'Cele_PhenotypicFeature_MolecularActivity_Monarch.csv'))
sub = PhenotypicFeature_MolecularActivity[PhenotypicFeature_MolecularActivity['Head_taxon_name'] == 'Danio rerio']
save(sub, os.path.join(OUT_ZEBRA, 'Zebra_PhenotypicFeature_MolecularActivity_Monarch.csv'))
sub = PhenotypicFeature_MolecularActivity[PhenotypicFeature_MolecularActivity['Head_taxon_name'] == 'Mus musculus']
save(sub, os.path.join(OUT_MOUSE, 'Mouse_PhenotypicFeature_MolecularActivity_Monarch.csv'))

Saved 368 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Human_PhenotypicFeature_MolecularActivity_Monarch.csv
Saved 4 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Celegans/Cele_PhenotypicFeature_MolecularActivity_Monarch.csv
Saved 455 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Zebrafish/Zebra_PhenotypicFeature_MolecularActivity_Monarch.csv
Saved 106 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Mouse/Mouse_PhenotypicFeature_MolecularActivity_Monarch.csv


/tmp/ipykernel_1542094/2805005729.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  PhenotypicFeature_MolecularActivity['Species_Species'] = (PhenotypicFeature_MolecularActivity['Head_taxon_name'] + '_' + PhenotypicFeature_MolecularActivity['Tail_taxon_name'].fillna(PhenotypicFeature_MolecularActivity['Head_taxon_name']))


---
## 16 · MolecularActivity_ChemicalEntity (F2)

In [64]:
MolecularActivity_ChemicalEntity = Edges[Edges['Relation'] == 'MolecularActivity_ChemicalEntity'].copy()
MolecularActivity_ChemicalEntity['Species_Species'] = (MolecularActivity_ChemicalEntity['Head_taxon_name'] + '_' +
    MolecularActivity_ChemicalEntity['Tail_taxon_name'].fillna(MolecularActivity_ChemicalEntity['Head_taxon_name']))
MolecularActivity_ChemicalEntity = MolecularActivity_ChemicalEntity.drop_duplicates(subset=['Head','Relation','Tail'])
MolecularActivity_ChemicalEntity = resolve_chebi_to_pubchem(MolecularActivity_ChemicalEntity, 'Tail')
MolecularActivity_ChemicalEntity['Head_taxon_name'] = 'Homo sapiens'
MolecularActivity_ChemicalEntity['Tail_taxon_name'] = 'Homo sapiens'
MolecularActivity_ChemicalEntity['Tail_ID_IS'] = 'Pubchem'
save(MolecularActivity_ChemicalEntity, os.path.join(OUT_HUMAN, 'Human_MolecularActivity_ChemicalEntity_Monarch.csv'))

  ChEBI Tail -> PubChem: 17,648 kept / 3,471 dropped
Saved 17,648 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Human_MolecularActivity_ChemicalEntity_Monarch.csv


/tmp/ipykernel_1542094/790560502.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  MolecularActivity_ChemicalEntity['Tail_taxon_name'].fillna(MolecularActivity_ChemicalEntity['Head_taxon_name']))


---
## 17 · SequenceVariant_Disease (F2)

In [65]:
SequenceVariant_Disease = Edges[Edges['Relation'] == 'SequenceVariant_Disease'].copy()
SequenceVariant_Disease['Species_Species'] = (SequenceVariant_Disease['Head_taxon_name'] + '_' +
    SequenceVariant_Disease['Tail_taxon_name'].fillna(SequenceVariant_Disease['Head_taxon_name']))
SequenceVariant_Disease = SequenceVariant_Disease.drop_duplicates(subset=['Head','Relation','Tail'])
# Parse protein name and mutation from variant label: 'GENE(p.VariantXXX)'
SequenceVariant_Disease['Head_name_Protein'] = SequenceVariant_Disease['Head_name'].str.extract(r'\((.*?)\)')
SequenceVariant_Disease['Head_name_mutaion'] = SequenceVariant_Disease['Head_name'].str.extract(r'\(p\.(.*?)\)')[0]
SequenceVariant_Disease['Head_name_mutaion'] = SequenceVariant_Disease['Head_name_mutaion'].apply(lambda x: f'p.{x}' if pd.notnull(x) else x)
SequenceVariant_Disease['Head_name_Protein_uniprot_mapped'] = SequenceVariant_Disease['Head_name_Protein'].map(Uniprot_gene_name_dict)
SequenceVariant_Disease['Combined'] = (SequenceVariant_Disease['Head_name_Protein'] + '_' +
    SequenceVariant_Disease['Head_name_mutaion']).fillna(SequenceVariant_Disease['Head_name'])
SequenceVariant_Disease[['Head','Combined']] = SequenceVariant_Disease[['Combined','Head']]
SequenceVariant_Disease = SequenceVariant_Disease.rename(columns={'Combined': 'Head_Orignal'})
# Resolve disease Tail to DOID or MONDO
SequenceVariant_Disease['Tail_DO_name'] = (SequenceVariant_Disease['Tail_name'].map(DOID_disname_dict)
    .fillna(SequenceVariant_Disease['Tail_name'].map(MONDO_dict)).fillna(SequenceVariant_Disease['Tail']))
SequenceVariant_Disease[['Tail','Tail_DO_name']] = SequenceVariant_Disease[['Tail_DO_name','Tail']]
SequenceVariant_Disease['Tail_ID_IS'] = np.where(SequenceVariant_Disease['Tail'].isna(), None,
    np.where(SequenceVariant_Disease['Tail'].str.startswith('DOID'), 'DOID', 'MONDO'))
save(SequenceVariant_Disease, os.path.join(OUT_HUMAN, 'Human_SequenceVariant_Disease_Monarch.csv'))

Saved 14,649 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Human_SequenceVariant_Disease_Monarch.csv


---
## 18 · Gene_Disease (F2)

In [66]:
Gene_Disease = Edges[Edges['Relation'] == 'Gene_Disease'].copy()
Gene_Disease['Species_Species'] = (Gene_Disease['Head_taxon_name'] + '_' +
    Gene_Disease['Tail_taxon_name'].fillna(Gene_Disease['Head_taxon_name']))
Gene_Disease = Gene_Disease.drop_duplicates(subset=['Head','Relation','Tail'])
Gene_Disease['Tail_DO_name'] = (Gene_Disease['Tail_name'].map(DOID_disname_dict)
    .fillna(Gene_Disease['Tail_name'].map(MONDO_dict)).fillna(Gene_Disease['Tail']))
Gene_Disease[['Tail','Tail_DO_name']] = Gene_Disease[['Tail_DO_name','Tail']]
Gene_Disease['Tail_ID_IS'] = np.where(Gene_Disease['Tail'].isna(), None,
    np.where(Gene_Disease['Tail'].str.startswith('DOID'), 'DOID', 'MONDO'))
Gene_Disease = resolve_gene_symbol_head(Gene_Disease)
Gene_Disease = select_cols(Gene_Disease)
save(Gene_Disease, os.path.join(OUT_HUMAN, 'Human_Gene_Disease_Monarch.csv'))

Saved 12,878 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Human_Gene_Disease_Monarch.csv


---
## 19 · Simple relation types (dedup + save, no additional annotation)

In [67]:
MolecularActivity_MolecularActivity = Edges[Edges['Relation'] == 'MolecularActivity_MolecularActivity'].copy()
MolecularActivity_MolecularActivity['Species_Species'] = MolecularActivity_MolecularActivity['Head_taxon_name'] + '_' + MolecularActivity_MolecularActivity['Tail_taxon_name'].fillna(MolecularActivity_MolecularActivity['Head_taxon_name'])
MolecularActivity_MolecularActivity = MolecularActivity_MolecularActivity.drop_duplicates(subset=['Head','Relation','Tail'])
save(MolecularActivity_MolecularActivity, os.path.join(OUT_PATH, 'Human', 'Human_MolecularActivity_MolecularActivity_Monarch.csv'))

Saved 12,350 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Human_MolecularActivity_MolecularActivity_Monarch.csv


/tmp/ipykernel_1542094/2793213299.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  MolecularActivity_MolecularActivity['Species_Species'] = MolecularActivity_MolecularActivity['Head_taxon_name'] + '_' + MolecularActivity_MolecularActivity['Tail_taxon_name'].fillna(MolecularActivity_MolecularActivity['Head_taxon_name'])


In [68]:
BiologicalProcess_ChemicalEntity = Edges[Edges['Relation'] == 'BiologicalProcess_ChemicalEntity'].copy()
BiologicalProcess_ChemicalEntity['Species_Species'] = BiologicalProcess_ChemicalEntity['Head_taxon_name'] + '_' + BiologicalProcess_ChemicalEntity['Tail_taxon_name'].fillna(BiologicalProcess_ChemicalEntity['Head_taxon_name'])
BiologicalProcess_ChemicalEntity = BiologicalProcess_ChemicalEntity.drop_duplicates(subset=['Head','Relation','Tail'])
save(BiologicalProcess_ChemicalEntity, os.path.join(OUT_PATH, 'Human', 'Human_BiologicalProcess_ChemicalEntity_Monarch.csv'))

Saved 4,560 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Human_BiologicalProcess_ChemicalEntity_Monarch.csv


/tmp/ipykernel_1542094/1097728301.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  BiologicalProcess_ChemicalEntity['Species_Species'] = BiologicalProcess_ChemicalEntity['Head_taxon_name'] + '_' + BiologicalProcess_ChemicalEntity['Tail_taxon_name'].fillna(BiologicalProcess_ChemicalEntity['Head_taxon_name'])


In [69]:
CellularComponent_CellularComponent = Edges[Edges['Relation'] == 'CellularComponent_CellularComponent'].copy()
CellularComponent_CellularComponent['Species_Species'] = CellularComponent_CellularComponent['Head_taxon_name'] + '_' + CellularComponent_CellularComponent['Tail_taxon_name'].fillna(CellularComponent_CellularComponent['Head_taxon_name'])
CellularComponent_CellularComponent = CellularComponent_CellularComponent.drop_duplicates(subset=['Head','Relation','Tail'])
save(CellularComponent_CellularComponent, os.path.join(OUT_PATH, 'Human', 'Human_CellularComponent_CellularComponent_Monarch.csv'))

Saved 6,627 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Human_CellularComponent_CellularComponent_Monarch.csv


/tmp/ipykernel_1542094/3588124422.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  CellularComponent_CellularComponent['Species_Species'] = CellularComponent_CellularComponent['Head_taxon_name'] + '_' + CellularComponent_CellularComponent['Tail_taxon_name'].fillna(CellularComponent_CellularComponent['Head_taxon_name'])


In [70]:
BiologicalProcess_CellularComponent = Edges[Edges['Relation'] == 'BiologicalProcess_CellularComponent'].copy()
BiologicalProcess_CellularComponent['Species_Species'] = BiologicalProcess_CellularComponent['Head_taxon_name'] + '_' + BiologicalProcess_CellularComponent['Tail_taxon_name'].fillna(BiologicalProcess_CellularComponent['Head_taxon_name'])
BiologicalProcess_CellularComponent = BiologicalProcess_CellularComponent.drop_duplicates(subset=['Head','Relation','Tail'])
save(BiologicalProcess_CellularComponent, os.path.join(OUT_PATH, 'Human', 'BiologicalProcess_CellularComponent_Monarch.csv'))

Saved 1,489 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/BiologicalProcess_CellularComponent_Monarch.csv


/tmp/ipykernel_1542094/1944589225.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  BiologicalProcess_CellularComponent['Species_Species'] = BiologicalProcess_CellularComponent['Head_taxon_name'] + '_' + BiologicalProcess_CellularComponent['Tail_taxon_name'].fillna(BiologicalProcess_CellularComponent['Head_taxon_name'])


In [71]:
BiologicalProcess_AnatomicalEntity = Edges[Edges['Relation'] == 'BiologicalProcess_AnatomicalEntity'].copy()
BiologicalProcess_AnatomicalEntity['Species_Species'] = BiologicalProcess_AnatomicalEntity['Head_taxon_name'] + '_' + BiologicalProcess_AnatomicalEntity['Tail_taxon_name'].fillna(BiologicalProcess_AnatomicalEntity['Head_taxon_name'])
BiologicalProcess_AnatomicalEntity = BiologicalProcess_AnatomicalEntity.drop_duplicates(subset=['Head','Relation','Tail'])
save(BiologicalProcess_AnatomicalEntity, os.path.join(OUT_PATH, 'Human', 'BiologicalProcess_AnatomicalEntity_Monarch.csv'))

Saved 1,300 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/BiologicalProcess_AnatomicalEntity_Monarch.csv


/tmp/ipykernel_1542094/4280791116.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  BiologicalProcess_AnatomicalEntity['Species_Species'] = BiologicalProcess_AnatomicalEntity['Head_taxon_name'] + '_' + BiologicalProcess_AnatomicalEntity['Tail_taxon_name'].fillna(BiologicalProcess_AnatomicalEntity['Head_taxon_name'])


In [72]:
MolecularActivity_BiologicalProcess = Edges[Edges['Relation'] == 'MolecularActivity_BiologicalProcess'].copy()
MolecularActivity_BiologicalProcess['Species_Species'] = MolecularActivity_BiologicalProcess['Head_taxon_name'] + '_' + MolecularActivity_BiologicalProcess['Tail_taxon_name'].fillna(MolecularActivity_BiologicalProcess['Head_taxon_name'])
MolecularActivity_BiologicalProcess = MolecularActivity_BiologicalProcess.drop_duplicates(subset=['Head','Relation','Tail'])
save(MolecularActivity_BiologicalProcess, os.path.join(OUT_PATH, 'Human', 'MolecularActivity_BiologicalProcess_Monarch.csv'))

Saved 848 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/MolecularActivity_BiologicalProcess_Monarch.csv


/tmp/ipykernel_1542094/2010655414.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  MolecularActivity_BiologicalProcess['Species_Species'] = MolecularActivity_BiologicalProcess['Head_taxon_name'] + '_' + MolecularActivity_BiologicalProcess['Tail_taxon_name'].fillna(MolecularActivity_BiologicalProcess['Head_taxon_name'])


In [73]:
BiologicalProcess_MolecularActivity = Edges[Edges['Relation'] == 'BiologicalProcess_MolecularActivity'].copy()
BiologicalProcess_MolecularActivity['Species_Species'] = BiologicalProcess_MolecularActivity['Head_taxon_name'] + '_' + BiologicalProcess_MolecularActivity['Tail_taxon_name'].fillna(BiologicalProcess_MolecularActivity['Head_taxon_name'])
BiologicalProcess_MolecularActivity = BiologicalProcess_MolecularActivity.drop_duplicates(subset=['Head','Relation','Tail'])
save(BiologicalProcess_MolecularActivity, os.path.join(OUT_PATH, 'Human', 'BiologicalProcess_MolecularActivity_Monarch.csv'))

Saved 834 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/BiologicalProcess_MolecularActivity_Monarch.csv


/tmp/ipykernel_1542094/2534806845.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  BiologicalProcess_MolecularActivity['Species_Species'] = BiologicalProcess_MolecularActivity['Head_taxon_name'] + '_' + BiologicalProcess_MolecularActivity['Tail_taxon_name'].fillna(BiologicalProcess_MolecularActivity['Head_taxon_name'])


In [74]:
PhenotypicFeature_Disease = Edges[Edges['Relation'] == 'PhenotypicFeature_Disease'].copy()
PhenotypicFeature_Disease['Species_Species'] = PhenotypicFeature_Disease['Head_taxon_name'] + '_' + PhenotypicFeature_Disease['Tail_taxon_name'].fillna(PhenotypicFeature_Disease['Head_taxon_name'])
PhenotypicFeature_Disease = PhenotypicFeature_Disease.drop_duplicates(subset=['Head','Relation','Tail'])
save(PhenotypicFeature_Disease, os.path.join(OUT_PATH, 'Human', 'PhenotypicFeature_Disease_Monarch.csv'))

Saved 757 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/PhenotypicFeature_Disease_Monarch.csv


/tmp/ipykernel_1542094/3379256764.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  PhenotypicFeature_Disease['Species_Species'] = PhenotypicFeature_Disease['Head_taxon_name'] + '_' + PhenotypicFeature_Disease['Tail_taxon_name'].fillna(PhenotypicFeature_Disease['Head_taxon_name'])


In [75]:
CellularComponent_MolecularActivity = Edges[Edges['Relation'] == 'CellularComponent_MolecularActivity'].copy()
CellularComponent_MolecularActivity['Species_Species'] = CellularComponent_MolecularActivity['Head_taxon_name'] + '_' + CellularComponent_MolecularActivity['Tail_taxon_name'].fillna(CellularComponent_MolecularActivity['Head_taxon_name'])
CellularComponent_MolecularActivity = CellularComponent_MolecularActivity.drop_duplicates(subset=['Head','Relation','Tail'])
save(CellularComponent_MolecularActivity, os.path.join(OUT_PATH, 'Human', 'CellularComponent_MolecularActivity_Monarch.csv'))

Saved 492 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/CellularComponent_MolecularActivity_Monarch.csv


/tmp/ipykernel_1542094/2794170609.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  CellularComponent_MolecularActivity['Species_Species'] = CellularComponent_MolecularActivity['Head_taxon_name'] + '_' + CellularComponent_MolecularActivity['Tail_taxon_name'].fillna(CellularComponent_MolecularActivity['Head_taxon_name'])


In [76]:
Protein_BiologicalProcess = Edges[Edges['Relation'] == 'Protein_BiologicalProcess'].copy()
Protein_BiologicalProcess['Species_Species'] = Protein_BiologicalProcess['Head_taxon_name'] + '_' + Protein_BiologicalProcess['Tail_taxon_name'].fillna(Protein_BiologicalProcess['Head_taxon_name'])
Protein_BiologicalProcess = Protein_BiologicalProcess.drop_duplicates(subset=['Head','Relation','Tail'])
save(Protein_BiologicalProcess, os.path.join(OUT_PATH, 'Human', 'Protein_BiologicalProcess_Monarch.csv'))

Saved 391 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Protein_BiologicalProcess_Monarch.csv


/tmp/ipykernel_1542094/826942602.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Protein_BiologicalProcess['Species_Species'] = Protein_BiologicalProcess['Head_taxon_name'] + '_' + Protein_BiologicalProcess['Tail_taxon_name'].fillna(Protein_BiologicalProcess['Head_taxon_name'])


In [77]:
MolecularEntity_CellularComponent = Edges[Edges['Relation'] == 'MolecularEntity_CellularComponent'].copy()
MolecularEntity_CellularComponent['Species_Species'] = MolecularEntity_CellularComponent['Head_taxon_name'] + '_' + MolecularEntity_CellularComponent['Tail_taxon_name'].fillna(MolecularEntity_CellularComponent['Head_taxon_name'])
MolecularEntity_CellularComponent = MolecularEntity_CellularComponent.drop_duplicates(subset=['Head','Relation','Tail'])
save(MolecularEntity_CellularComponent, os.path.join(OUT_PATH, 'Human', 'MolecularEntity_CellularComponent_Monarch.csv'))

Saved 359 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/MolecularEntity_CellularComponent_Monarch.csv


/tmp/ipykernel_1542094/1809969225.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  MolecularEntity_CellularComponent['Species_Species'] = MolecularEntity_CellularComponent['Head_taxon_name'] + '_' + MolecularEntity_CellularComponent['Tail_taxon_name'].fillna(MolecularEntity_CellularComponent['Head_taxon_name'])


In [78]:
CellularComponent_BiologicalProcess = Edges[Edges['Relation'] == 'CellularComponent_BiologicalProcess'].copy()
CellularComponent_BiologicalProcess['Species_Species'] = CellularComponent_BiologicalProcess['Head_taxon_name'] + '_' + CellularComponent_BiologicalProcess['Tail_taxon_name'].fillna(CellularComponent_BiologicalProcess['Head_taxon_name'])
CellularComponent_BiologicalProcess = CellularComponent_BiologicalProcess.drop_duplicates(subset=['Head','Relation','Tail'])
save(CellularComponent_BiologicalProcess, os.path.join(OUT_PATH, 'Human', 'CellularComponent_BiologicalProcess_Monarch.csv'))

Saved 323 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/CellularComponent_BiologicalProcess_Monarch.csv


/tmp/ipykernel_1542094/3846831366.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  CellularComponent_BiologicalProcess['Species_Species'] = CellularComponent_BiologicalProcess['Head_taxon_name'] + '_' + CellularComponent_BiologicalProcess['Tail_taxon_name'].fillna(CellularComponent_BiologicalProcess['Head_taxon_name'])


In [79]:
Disease_MolecularEntity = Edges[Edges['Relation'] == 'Disease_MolecularEntity'].copy()
Disease_MolecularEntity['Species_Species'] = Disease_MolecularEntity['Head_taxon_name'] + '_' + Disease_MolecularEntity['Tail_taxon_name'].fillna(Disease_MolecularEntity['Head_taxon_name'])
Disease_MolecularEntity = Disease_MolecularEntity.drop_duplicates(subset=['Head','Relation','Tail'])
save(Disease_MolecularEntity, os.path.join(OUT_PATH, 'Human', 'Disease_MolecularEntity_Monarch.csv'))

Saved 303 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Disease_MolecularEntity_Monarch.csv


/tmp/ipykernel_1542094/400867615.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Disease_MolecularEntity['Species_Species'] = Disease_MolecularEntity['Head_taxon_name'] + '_' + Disease_MolecularEntity['Tail_taxon_name'].fillna(Disease_MolecularEntity['Head_taxon_name'])


In [80]:
MolecularActivity_Protein = Edges[Edges['Relation'] == 'MolecularActivity_Protein'].copy()
MolecularActivity_Protein['Species_Species'] = MolecularActivity_Protein['Head_taxon_name'] + '_' + MolecularActivity_Protein['Tail_taxon_name'].fillna(MolecularActivity_Protein['Head_taxon_name'])
MolecularActivity_Protein = MolecularActivity_Protein.drop_duplicates(subset=['Head','Relation','Tail'])
save(MolecularActivity_Protein, os.path.join(OUT_PATH, 'Human', 'MolecularActivity_Protein_Monarch.csv'))

Saved 294 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/MolecularActivity_Protein_Monarch.csv


/tmp/ipykernel_1542094/2209507493.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  MolecularActivity_Protein['Species_Species'] = MolecularActivity_Protein['Head_taxon_name'] + '_' + MolecularActivity_Protein['Tail_taxon_name'].fillna(MolecularActivity_Protein['Head_taxon_name'])


In [81]:
BiologicalProcess_MolecularEntity = Edges[Edges['Relation'] == 'BiologicalProcess_MolecularEntity'].copy()
BiologicalProcess_MolecularEntity['Species_Species'] = BiologicalProcess_MolecularEntity['Head_taxon_name'] + '_' + BiologicalProcess_MolecularEntity['Tail_taxon_name'].fillna(BiologicalProcess_MolecularEntity['Head_taxon_name'])
BiologicalProcess_MolecularEntity = BiologicalProcess_MolecularEntity.drop_duplicates(subset=['Head','Relation','Tail'])
save(BiologicalProcess_MolecularEntity, os.path.join(OUT_PATH, 'Human', 'BiologicalProcess_MolecularEntity_Monarch.csv'))

Saved 50 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/BiologicalProcess_MolecularEntity_Monarch.csv


/tmp/ipykernel_1542094/2766544760.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  BiologicalProcess_MolecularEntity['Species_Species'] = BiologicalProcess_MolecularEntity['Head_taxon_name'] + '_' + BiologicalProcess_MolecularEntity['Tail_taxon_name'].fillna(BiologicalProcess_MolecularEntity['Head_taxon_name'])


In [82]:
BiologicalProcess_Protein = Edges[Edges['Relation'] == 'BiologicalProcess_Protein'].copy()
BiologicalProcess_Protein['Species_Species'] = BiologicalProcess_Protein['Head_taxon_name'] + '_' + BiologicalProcess_Protein['Tail_taxon_name'].fillna(BiologicalProcess_Protein['Head_taxon_name'])
BiologicalProcess_Protein = BiologicalProcess_Protein.drop_duplicates(subset=['Head','Relation','Tail'])
save(BiologicalProcess_Protein, os.path.join(OUT_PATH, 'Human', 'BiologicalProcess_Protein_Monarch.csv'))

Saved 28 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/BiologicalProcess_Protein_Monarch.csv


/tmp/ipykernel_1542094/579527661.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  BiologicalProcess_Protein['Species_Species'] = BiologicalProcess_Protein['Head_taxon_name'] + '_' + BiologicalProcess_Protein['Tail_taxon_name'].fillna(BiologicalProcess_Protein['Head_taxon_name'])


---
## 20 · Disease sections with Head DOID/MONDO resolution

In [83]:
ChemicalEntity_Disease = Edges[Edges['Relation'] == 'ChemicalEntity_Disease'].copy()
ChemicalEntity_Disease['Species_Species'] = ChemicalEntity_Disease['Head_taxon_name'] + '_' + ChemicalEntity_Disease['Tail_taxon_name'].fillna(ChemicalEntity_Disease['Head_taxon_name'])
ChemicalEntity_Disease = ChemicalEntity_Disease.drop_duplicates(subset=['Head','Relation','Tail'])
ChemicalEntity_Disease = resolve_disease_head(ChemicalEntity_Disease)
ChemicalEntity_Disease = resolve_chebi_to_pubchem(ChemicalEntity_Disease, 'Head')
save(ChemicalEntity_Disease, os.path.join(OUT_PATH, 'Human', 'Human_ChemicalEntity_Disease_Monarch.csv'))

  ChEBI Head -> PubChem: 6,789 kept / 1,833 dropped
Saved 6,789 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Human_ChemicalEntity_Disease_Monarch.csv


/tmp/ipykernel_1542094/2559279464.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ChemicalEntity_Disease['Species_Species'] = ChemicalEntity_Disease['Head_taxon_name'] + '_' + ChemicalEntity_Disease['Tail_taxon_name'].fillna(ChemicalEntity_Disease['Head_taxon_name'])
/tmp/ipykernel_1542094/3713943533.py:51: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(df['Head_name'].map(MONDO_dict))


In [84]:
Disease_AnatomicalEntity = Edges[Edges['Relation'] == 'Disease_AnatomicalEntity'].copy()
Disease_AnatomicalEntity['Species_Species'] = Disease_AnatomicalEntity['Head_taxon_name'] + '_' + Disease_AnatomicalEntity['Tail_taxon_name'].fillna(Disease_AnatomicalEntity['Head_taxon_name'])
Disease_AnatomicalEntity = Disease_AnatomicalEntity.drop_duplicates(subset=['Head','Relation','Tail'])
Disease_AnatomicalEntity = resolve_disease_head(Disease_AnatomicalEntity)
save(Disease_AnatomicalEntity, os.path.join(OUT_PATH, 'Human', 'Disease_AnatomicalEntity_Monarch.csv'))

Saved 1,016 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Disease_AnatomicalEntity_Monarch.csv


/tmp/ipykernel_1542094/2922926038.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Disease_AnatomicalEntity['Species_Species'] = Disease_AnatomicalEntity['Head_taxon_name'] + '_' + Disease_AnatomicalEntity['Tail_taxon_name'].fillna(Disease_AnatomicalEntity['Head_taxon_name'])


In [85]:
Disease_BiologicalProcess = Edges[Edges['Relation'] == 'Disease_BiologicalProcess'].copy()
Disease_BiologicalProcess['Species_Species'] = Disease_BiologicalProcess['Head_taxon_name'] + '_' + Disease_BiologicalProcess['Tail_taxon_name'].fillna(Disease_BiologicalProcess['Head_taxon_name'])
Disease_BiologicalProcess = Disease_BiologicalProcess.drop_duplicates(subset=['Head','Relation','Tail'])
Disease_BiologicalProcess = resolve_disease_head(Disease_BiologicalProcess)
save(Disease_BiologicalProcess, os.path.join(OUT_PATH, 'Human', 'Disease_BiologicalProcess_Monarch.csv'))

Saved 246 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Disease_BiologicalProcess_Monarch.csv


/tmp/ipykernel_1542094/1338224163.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Disease_BiologicalProcess['Species_Species'] = Disease_BiologicalProcess['Head_taxon_name'] + '_' + Disease_BiologicalProcess['Tail_taxon_name'].fillna(Disease_BiologicalProcess['Head_taxon_name'])


In [86]:
Disease_MolecularActivity = Edges[Edges['Relation'] == 'Disease_MolecularActivity'].copy()
Disease_MolecularActivity['Species_Species'] = Disease_MolecularActivity['Head_taxon_name'] + '_' + Disease_MolecularActivity['Tail_taxon_name'].fillna(Disease_MolecularActivity['Head_taxon_name'])
Disease_MolecularActivity = Disease_MolecularActivity.drop_duplicates(subset=['Head','Relation','Tail'])
Disease_MolecularActivity = resolve_disease_head(Disease_MolecularActivity)
save(Disease_MolecularActivity, os.path.join(OUT_PATH, 'Human', 'Disease_MolecularActivity_Monarch.csv'))

Saved 103 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Disease_MolecularActivity_Monarch.csv


/tmp/ipykernel_1542094/927312143.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Disease_MolecularActivity['Species_Species'] = Disease_MolecularActivity['Head_taxon_name'] + '_' + Disease_MolecularActivity['Tail_taxon_name'].fillna(Disease_MolecularActivity['Head_taxon_name'])


In [87]:
Disease_ChemicalEntity = Edges[Edges['Relation'] == 'Disease_ChemicalEntity'].copy()
Disease_ChemicalEntity['Species_Species'] = Disease_ChemicalEntity['Head_taxon_name'] + '_' + Disease_ChemicalEntity['Tail_taxon_name'].fillna(Disease_ChemicalEntity['Head_taxon_name'])
Disease_ChemicalEntity = Disease_ChemicalEntity.drop_duplicates(subset=['Head','Relation','Tail'])
Disease_ChemicalEntity = resolve_disease_head(Disease_ChemicalEntity)
Disease_ChemicalEntity = resolve_chebi_to_pubchem(Disease_ChemicalEntity, 'Tail')
save(Disease_ChemicalEntity, os.path.join(OUT_PATH, 'Human', 'Disease_ChemicalEntity_Monarch.csv'))

  ChEBI Tail -> PubChem: 30 kept / 18 dropped
Saved 30 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Disease_ChemicalEntity_Monarch.csv


/tmp/ipykernel_1542094/4016449106.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Disease_ChemicalEntity['Species_Species'] = Disease_ChemicalEntity['Head_taxon_name'] + '_' + Disease_ChemicalEntity['Tail_taxon_name'].fillna(Disease_ChemicalEntity['Head_taxon_name'])


In [88]:
Disease_CellularComponent = Edges[Edges['Relation'] == 'Disease_CellularComponent'].copy()
Disease_CellularComponent['Species_Species'] = Disease_CellularComponent['Head_taxon_name'] + '_' + Disease_CellularComponent['Tail_taxon_name'].fillna(Disease_CellularComponent['Head_taxon_name'])
Disease_CellularComponent = Disease_CellularComponent.drop_duplicates(subset=['Head','Relation','Tail'])
Disease_CellularComponent = resolve_disease_head(Disease_CellularComponent)
save(Disease_CellularComponent, os.path.join(OUT_PATH, 'Human', 'Disease_CellularComponent_Monarch.csv'))

Saved 30 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Disease_CellularComponent_Monarch.csv


/tmp/ipykernel_1542094/4060035614.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Disease_CellularComponent['Species_Species'] = Disease_CellularComponent['Head_taxon_name'] + '_' + Disease_CellularComponent['Tail_taxon_name'].fillna(Disease_CellularComponent['Head_taxon_name'])


---
## 21 · AnatomicalEntity → CellularComponent / BiologicalProcess (F2)

Species inferred from UBERON (Human) and WBbt (Drosophila) prefixes.

In [89]:
for rel, label in [('AnatomicalEntity_CellularComponent','AnatomicalEntity_CellularComponent'),
                    ('AnatomicalEntity_BiologicalProcess','AnatomicalEntity_BiologicalProcess')]:
    df = Edges[Edges['Relation'] == rel].copy()
    df['Species_Species'] = df['Head_taxon_name'] + '_' + df['Tail_taxon_name'].fillna(df['Head_taxon_name'])
    df = df.drop_duplicates(subset=['Head','Relation','Tail'])
    Head_Src = {'WBbt':'Drosophila melanogaster','UBERON':'Homo sapiens'}
    df['Head_taxon_name'] = df['Head_ID_IS'].map(Head_Src).fillna(np.nan)
    df = df[~df['Head_taxon_name'].isna()]
    # BUG FIX: original Section 38 processed AnatomicalEntity_BiologicalProcess but
    # then saved AnatomicalEntity_CellularComponent again with wrong filename.
    for sp, var, out in [('Homo sapiens','Human','HUMAN'),('Drosophila melanogaster','Droso','DROSO')]:
        sub = df[df['Head_taxon_name'] == sp]
        save(sub, os.path.join(eval(f'OUT_{out}'), f'{var}_{label}_Monarch.csv'))

/tmp/ipykernel_1542094/4038694674.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Species_Species'] = df['Head_taxon_name'] + '_' + df['Tail_taxon_name'].fillna(df['Head_taxon_name'])


Saved 71 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Human_AnatomicalEntity_CellularComponent_Monarch.csv
Saved 2,766 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Drosophila/Droso_AnatomicalEntity_CellularComponent_Monarch.csv
Saved 321 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Human_AnatomicalEntity_BiologicalProcess_Monarch.csv
Saved 0 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Drosophila/Droso_AnatomicalEntity_BiologicalProcess_Monarch.csv


/tmp/ipykernel_1542094/4038694674.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Species_Species'] = df['Head_taxon_name'] + '_' + df['Tail_taxon_name'].fillna(df['Head_taxon_name'])


---
## 22 · AnatomicalEntity_ChemicalEntity (F2)

In [90]:
AnatomicalEntity_ChemicalEntity = Edges[Edges['Relation'] == 'AnatomicalEntity_ChemicalEntity'].copy()
AnatomicalEntity_ChemicalEntity['Species_Species'] = (AnatomicalEntity_ChemicalEntity['Head_taxon_name'] + '_' +
    AnatomicalEntity_ChemicalEntity['Tail_taxon_name'].fillna(AnatomicalEntity_ChemicalEntity['Head_taxon_name']))
AnatomicalEntity_ChemicalEntity = AnatomicalEntity_ChemicalEntity.drop_duplicates(subset=['Head','Relation','Tail'])
AnatomicalEntity_ChemicalEntity = resolve_chebi_to_pubchem(AnatomicalEntity_ChemicalEntity, 'Tail')
save(AnatomicalEntity_ChemicalEntity, os.path.join(OUT_HUMAN, 'AnatomicalEntity_ChemicalEntity_Monarch.csv'))

  ChEBI Tail -> PubChem: 9 kept / 22 dropped
Saved 9 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/AnatomicalEntity_ChemicalEntity_Monarch.csv


/tmp/ipykernel_1542094/3562482533.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  AnatomicalEntity_ChemicalEntity['Tail_taxon_name'].fillna(AnatomicalEntity_ChemicalEntity['Head_taxon_name']))


---
## 23 · SequenceVariant_PhenotypicFeature (F2)

In [91]:
SequenceVariant_PhenotypicFeature = Edges[Edges['Relation'] == 'SequenceVariant_PhenotypicFeature'].copy()
SequenceVariant_PhenotypicFeature['Species_Species'] = (SequenceVariant_PhenotypicFeature['Head_taxon_name'] + '_' +
    SequenceVariant_PhenotypicFeature['Tail_taxon_name'].fillna(SequenceVariant_PhenotypicFeature['Head_taxon_name']))
SequenceVariant_PhenotypicFeature = SequenceVariant_PhenotypicFeature.drop_duplicates(subset=['Head','Relation','Tail'])
SequenceVariant_PhenotypicFeature['Head_name_Protein'] = SequenceVariant_PhenotypicFeature['Head_name'].str.extract(r'\((.*?)\)')
SequenceVariant_PhenotypicFeature['Head_name_mutaion'] = SequenceVariant_PhenotypicFeature['Head_name'].str.extract(r'\(p\.(.*?)\)')[0]
SequenceVariant_PhenotypicFeature['Head_name_mutaion'] = SequenceVariant_PhenotypicFeature['Head_name_mutaion'].apply(lambda x: f'p.{x}' if pd.notnull(x) else x)
SequenceVariant_PhenotypicFeature['Head_name_Protein_uniprot_mapped'] = SequenceVariant_PhenotypicFeature['Head_name_Protein'].map(Uniprot_gene_name_dict)
SequenceVariant_PhenotypicFeature['Combined'] = (SequenceVariant_PhenotypicFeature['Head_name_Protein'] + '_' +
    SequenceVariant_PhenotypicFeature['Head_name_mutaion']).fillna(SequenceVariant_PhenotypicFeature['Head_name'])
SequenceVariant_PhenotypicFeature[['Head','Combined']] = SequenceVariant_PhenotypicFeature[['Combined','Head']]
SequenceVariant_PhenotypicFeature = SequenceVariant_PhenotypicFeature.rename(columns={'Combined': 'Head_Orignal'})
save(SequenceVariant_PhenotypicFeature, os.path.join(OUT_HUMAN, 'SequenceVariant_PhenotypicFeature_Monarch.csv'))

Saved 146 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/SequenceVariant_PhenotypicFeature_Monarch.csv


---
## 24 · Genotype_PhenotypicFeature (F2) — per species by ID prefix

In [92]:
Genotype_Disease = Edges[Edges['Relation'] == 'Genotype_PhenotypicFeature'].copy()
Genotype_Disease['Species_Species'] = (Genotype_Disease['Head_taxon_name'] + '_' +
    Genotype_Disease['Tail_taxon_name'].fillna(Genotype_Disease['Head_taxon_name']))
Genotype_Disease = Genotype_Disease.drop_duplicates(subset=['Head','Relation','Tail'])

Zebra_Genotype = Genotype_Disease[Genotype_Disease['Head_ID_IS'] == 'ZFIN']
Mouse_Genotype = Genotype_Disease[Genotype_Disease['Head_ID_IS'] == 'MGI']
Rat_Genotype   = Genotype_Disease[Genotype_Disease['Head_ID_IS'] == 'RGD']

save(Zebra_Genotype, os.path.join(OUT_ZEBRA, 'Zebra_Genotype_Disease_Monarch.csv'))
save(Mouse_Genotype, os.path.join(OUT_MOUSE, 'Mouse_Genotype_Disease_Monarch.csv'))
save(Rat_Genotype,   os.path.join(OUT_RAT,   'RAT_Genotype_Disease_Monarch.csv'))

Saved 133,736 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Zebrafish/Zebra_Genotype_Disease_Monarch.csv
Saved 383,639 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Mouse/Mouse_Genotype_Disease_Monarch.csv
Saved 1,835 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Rat/RAT_Genotype_Disease_Monarch.csv


---
## 25 · Summary — all output files

In [93]:
print(f"Output files under: {OUT_PATH}\n")
cols = ["Relation","Head_type","Tail_type","Source","KG_Source","Head_ID_IS","Tail_ID_IS"]
df_list = []
for file_path in sorted(glob(os.path.join(OUT_PATH, '**', '*.csv'), recursive=True)):
    try:
        total_df = pd.read_csv(file_path)
        total_rows = len(total_df)
        temp_df = total_df.head(5)
        available = [c for c in cols if c in temp_df.columns]
        row = {'Filename': os.path.relpath(file_path, OUT_PATH), 'no. of triples': total_rows}
        for c in available:
            row[c] = set(temp_df[c].dropna().tolist())
        df_list.append(pd.DataFrame([row]))
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
combined_df = pd.concat(df_list, ignore_index=True)
print(f"Total files: {len(combined_df)}")
combined_df

Output files under: /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/

Total files: 105


,Filename,no. of triples,Relation,Head_type,Tail_type,Head_ID_IS,Tail_ID_IS,KG_Source
0,Celegans/Cele_AnatomicalEntity_AnatomicalEntit...,15092,{AnatomicalEntity_AnatomicalEntity},{AnatomicalEntity},{AnatomicalEntity},{WBbt},{WBbt},NaN
1,Celegans/Cele_Gene_Pathway_MONARCH.csv,12698,{Gene_Pathway},{Gene},{Pathway},{WB},{Reactome},NaN
2,Celegans/Cele_Phenotype_CellularComponent_Mona...,76,{PhenotypicFeature_CellularComponent},{PhenotypicFeature},{CellularComponent},{WBPhenotype},{GO},NaN
3,Celegans/Cele_Phenotype_ChemicalEntity_Monarch...,11,{PhenotypicFeature_ChemicalEntity},{PhenotypicFeature},{ChemicalEntity},{WBPhenotype},{CHEBI},NaN
4,Celegans/Cele_PhenotypicFeature_BiologicalProc...,488,{PhenotypicFeature_BiologicalProcess},{PhenotypicFeature},{BiologicalProcess},{WBPhenotype},{GO},NaN
...,...,...,...,...,...,...,...,...
100,Zebrafish/Zebra_Genotype_Disease_Monarch.csv,133736,{Genotype_PhenotypicFeature},{Genotype},{PhenotypicFeature},{ZFIN},{ZP},NaN
101,Zebrafish/Zebra_Phenotype_CellularComponent_Mo...,5378,{PhenotypicFeature_CellularComponent},{PhenotypicFeature},{CellularComponent},{ZP},{GO},NaN
102,Zebrafish/Zebra_Phenotype_ChemicalEntity_Monar...,702,{PhenotypicFeature_ChemicalEntity},{PhenotypicFeature},{ChemicalEntity},{ZP},{CHEBI},NaN
103,Zebrafish/Zebra_PhenotypicFeature_BiologicalPr...,10674,{PhenotypicFeature_BiologicalProcess},{PhenotypicFeature},{BiologicalProcess},{ZP},{GO},NaN
